# Analysis

**Hypothesis**: Within ventricular and atrial fibroblast and endothelial populations, a discrete subset of cells with unusually high transcriptional and neighborhood diversity acts as spatial hubs at interfaces between cardiomyocyte domains, leading to a strong association between per-cell transcriptional complexity, local spatial cell-type heterogeneity, and reduced Purity that is not explained by UMI depth alone.

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

# Set up visualization defaults for better plots
sc.settings.verbosity = 3
sc.settings.figsize = (8, 8)
sc.settings.dpi = 100
sc.settings.facecolor = 'white'
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['savefig.dpi'] = 150
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.2)

# Load data
print("Loading data...")
adata = sc.read_h5ad("/home/mingqiam/TissueAgent/demo/data/dataset_farah_heart_merfish.h5ad")
print(f"Data loaded: {adata.shape[0]} cells and {adata.shape[1]} genes")


Loading data...


Data loaded: 228635 cells and 238 genes


# Analysis Plan

**Hypothesis**: Within ventricular and atrial fibroblast and endothelial populations, a discrete subset of cells with unusually high transcriptional and neighborhood diversity acts as spatial hubs at interfaces between cardiomyocyte domains, leading to a strong association between per-cell transcriptional complexity, local spatial cell-type heterogeneity, and reduced Purity that is not explained by UMI depth alone.

## Steps:
- Refine and robustify the initial AnnData inventory/QC summaries by explicitly handling non-numeric QC entries, reporting missingness for key annotations (UMI Count, Complexity, Purity, Populations), and summarizing QC metrics both globally and per-Populations category to establish baseline differences relevant to the hub/heterogeneity hypothesis.
- Quantify per-cell local neighborhood composition in physical space by building a k-nearest-neighbor graph on adata.obsm['spatial'] (for several k values) and, for each cell, computing Shannon entropy of neighboring Populations labels as a continuous spatial heterogeneity score stored in adata.obs.
- Within major non-cardiomyocyte populations (e.g. vFibro, aFibro, EPDC, BEC, VEC, VSMC, Pericyte), model the relationship between transcriptional Complexity and spatial heterogeneity (entropy) while adjusting for UMI Count and Purity using multiple linear regression or partial correlation, and test whether spatial heterogeneity remains a significant predictor of Complexity.
- Identify and characterize 'interface hub' cells within each focal non-cardiomyocyte population as those with high spatial heterogeneity (e.g. top 10% entropy) and intermediate-to-high Purity, and compare their UMI Count and Complexity to non-hub cells using appropriate non-parametric tests (e.g., Mann–Whitney U) with multiple-testing-aware reporting.
- Within each focal non-cardiomyocyte population, perform differential expression analysis between interface hub and non-hub cells using sc.tl.rank_genes_groups (method='wilcoxon'), and summarize for each population the top marker genes that distinguish putative hub states, focusing on known signaling/ligand genes present in the panel.
- Assess whether interface hub cells are spatially enriched at boundaries between ventricular and atrial cardiomyocyte territories by computing, for each non-cardiomyocyte cell, its minimum spatial distance to the nearest ventricular versus atrial cardiomyocyte and comparing these distances between hub and non-hub cells within each population using statistical tests and text-based summaries.


## Refines the initial AnnData inventory and QC step by robustly converting QC columns to numeric with missingness reporting, adding per-population QC summaries, and safely checking embedding ranges, all via text output to prepare for downstream spatial heterogeneity and hub analyses.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

# 1. Basic AnnData inventory
print("AnnData object summary:\n")
print(adata)

print("\n.obs columns:\n", adata.obs.columns.tolist())
print("\n.var columns:\n", adata.var.columns.tolist())
print("\n.obsm keys:\n", list(adata.obsm.keys()))
print("\n.uns keys:\n", list(adata.uns.keys()))

# 2. Basic QC summaries for UMI Count, Complexity, Purity with robust numeric casting
qc_cols = [col for col in ["UMI Count", "Complexity", "Purity"] if col in adata.obs.columns]

print("\nQC variable summaries (per-column):")
for col in qc_cols:
    series = pd.to_numeric(adata.obs[col], errors="coerce")
    n_na = series.isna().sum()
    if n_na > 0:
        print(f"\n{n_na} cells have non-numeric or missing values for {col} (excluded from numeric summary).")
    desc = series.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
    print(f"\nSummary for {col}:")
    print(desc.to_string())

# 3. Pairwise Pearson and Spearman correlations among QC metrics
if len(qc_cols) >= 2:
    qc_df = adata.obs[qc_cols].apply(pd.to_numeric, errors="coerce")
    print("\nPearson correlation matrix among QC metrics:")
    print(qc_df.corr(method="pearson").to_string())

    print("\nSpearman correlation matrix among QC metrics:")
    print(qc_df.corr(method="spearman").to_string())
else:
    print("\nFewer than two QC metrics found; skipping correlation analysis.")

# 4. Population and sample summaries with missingness
for cat in ["Populations", "Sample_ID", "Batch"]:
    if cat in adata.obs.columns:
        n_na_cat = adata.obs[cat].isna().sum()
        print(f"\nCategory counts for {cat} (missing: {n_na_cat} cells):")
        value_counts = adata.obs[cat].value_counts()
        print(value_counts.to_string())

# 5. Per-population QC summaries to understand baseline differences
if "Populations" in adata.obs.columns and qc_cols:
    qc_numeric = adata.obs[qc_cols].apply(pd.to_numeric, errors="coerce")
    tmp_df = qc_numeric.copy()
    tmp_df["Populations"] = adata.obs["Populations"].values
    print("\nPer-population QC means (rows: Populations, columns: QC metrics):")
    print(tmp_df.groupby("Populations")[qc_cols].mean().to_string(float_format=lambda x: f"{x:.3f}"))

# 6. Check embeddings presence and basic ranges
for key in ["X_umap", "spatial"]:
    if key in adata.obsm.keys():
        emb = adata.obsm[key]
        # Convert to dense array if sparse, to be safe
        try:
            emb_array = np.asarray(emb.todense())
        except AttributeError:
            emb_array = np.asarray(emb)
        print(f"\nEmbedding '{key}' present with shape {emb_array.shape}.")
        print(
            f"  Coordinate ranges: x in [{emb_array[:, 0].min():.3f}, {emb_array[:, 0].max():.3f}], "
            f"y in [{emb_array[:, 1].min():.3f}, {emb_array[:, 1].max():.3f}]"
        )

print("\nRefined inventory and QC summaries complete.")


AnnData object summary:

AnnData object with n_obs × n_vars = 228635 × 238
    obs: 'Sample_ID', 'Batch', 'UMI Count', 'leiden', 'Complexity', 'Populations', 'Purity'
    uns: 'dataset_provenance'
    obsm: 'X_umap', 'spatial'

.obs columns:
 ['Sample_ID', 'Batch', 'UMI Count', 'leiden', 'Complexity', 'Populations', 'Purity']

.var columns:
 []

.obsm keys:
 ['X_umap', 'spatial']

.uns keys:
 ['dataset_provenance']

QC variable summaries (per-column):

Summary for UMI Count:
count    228635.000000
mean        442.819717
std         284.013686
min           9.000000
5%           98.000000
25%         237.000000
50%         386.000000
75%         583.000000
95%         983.000000
max        5648.000000

Summary for Complexity:
count    228635.00000
mean          9.88252
std           2.89125
min           1.00000
5%            5.00000
25%           8.00000
50%          10.00000
75%          12.00000
95%          15.00000
max          20.00000

Summary for Purity:
count    228635.000000
m

### Agent Interpretation

These QC summaries are a good foundation and already give some signals that are directly relevant to the “hub / heterogeneity” hypothesis.

Key points and how they should shape the next steps:

1. **Complexity vs Purity vs UMI Count: baseline structure**
   - Complexity is **only weakly anti-correlated with UMI Count** (Pearson ~ –0.21, Spearman ~ –0.18). This is helpful: transcriptional complexity is *not* just a UMI-depth proxy, which is exactly what we need for the hypothesis (we want to argue that complexity reflects biology rather than sequencing depth).
   - Complexity is **strongly and negatively correlated with Purity** (Pearson –0.57, Spearman –0.52). This is a big red flag for interpretation: high “Complexity” is likely dominated by mixed barcodes / local multi-cell capture or partial segmentation errors (or, more generally, multi-identity pixels) rather than a subtle “hub state.”
   - Purity is basically uncorrelated with UMI Count (Pearson ~0.02, Spearman ~0): so the purity metric is not trivially tracking depth and likely captures spatial/segmentation aspects.

   Implication for hypothesis testing:
   - You **must explicitly adjust for Purity** whenever you look at Complexity. If not, any association between Complexity and spatial heterogeneity will be hard to interpret as “hub biology.”
   - It also justifies the planned design criterion of restricting “hub” cells to *intermediate-to-high Purity*; otherwise you will simply be picking up obvious doublets / mixed pixels.

2. **Per-population QC differences: where to look for hubs**
   The per-population means already suggest which non-cardiomyocyte groups are most promising:

   - **Fibroblasts**
     - vFibro: UMI ~413, Complexity ~10.7, Purity ~0.42
     - aFibro: UMI ~360, Complexity ~8.34, Purity ~0.58
     - adFibro: UMI ~223, Complexity ~7.79, Purity ~0.50
     These are good primary targets given the hypothesis. Note that **ventricular fibroblasts are notably lower purity and higher complexity than atrial fibroblasts**, which might bias interface detection if purity varies by region. Any ventriculo–atrial pattern you see later could reflect this baseline difference.

   - **Endothelial-related populations**
     - BEC: Complexity ~10.7, Purity ~0.45
     - VEC: Complexity ~8.56, Purity ~0.54
     - vEndocardial: Complexity ~9.84, Purity ~0.44
     - aEndocardial: Complexity ~5.81, Purity ~0.66
     Again, striking **atria–ventricle contrasts**: atrial endocardial cells are higher purity and lower complexity than ventricular ones. This is biologically interesting but also a potential confound when comparing “interface” vs “core” behavior.

   - **Others (pericytes, VSMC, VIC, EPDC)**
     - EPDC: high Complexity (~12.9) with low Purity (~0.45), similar pattern to other low-purity, high-complexity groups.
     - VIC, VSMC: relatively high Purity (~0.62–0.64) with moderate Complexity (~8.9–9.2); these are actually attractive for isolating high-complexity, **high-purity** outliers, because their baseline purity is good and complexity is not at ceiling.

   Implication for upcoming steps:
   - For the core hypothesis (within vFibro, aFibro, EPDC, BEC, VEC, VSMC, Pericyte), you should treat each population’s **baseline (mean, variance) of Purity and Complexity** as part of the model design. Interface hub thresholds based on global quantiles might not be appropriate; consider population-specific quantiles for entropy and maybe for complexity if you later stratify by that.
   - Especially for EPDC, BEC, vFibro, and vEndocardial, expect that any high-complexity subset will be almost indistinguishable from low-purity cells unless you tightly enforce a purity cutoff.

3. **How these results constrain / support the hub hypothesis**
   - The current data are **compatible** with the idea that some cells will show **high complexity not explained by UMI depth**, since the UMI–Complexity correlation is modest.
   - However, Complexity and Purity are so tightly linked that you will need to show **an additional signal beyond that axis**:
     - Either: within *medium-to-high purity* cells, Complexity remains associated with spatial neighborhood entropy.
     - Or: even at fixed Purity and UMI Count, neighborhood entropy significantly predicts Complexity (as you already plan with regression / partial correlation).

   Put differently: the QC structure is consistent with a mix of:
   - “impure/mixed” pixels with high complexity and low purity; and
   - relatively pure pixels with moderate complexity.
   The hypothesis will only be convincing if “hub” cells look like **the latter with unusually high local heterogeneity**, not like obvious impurity artifacts.

4. **Recommendations for the next analysis steps**

   For Step 2 (spatial entropy / heterogeneity):
   - Proceed with computing kNN-based neighborhood entropy on `.obsm["spatial"]` for several k (e.g., 10, 20, 50). Use **Populations** as the categorical labels for entropy.
   - Given the sample size, consider **computing entropy separately per sample** or at least making sure the kNN graph is *not* inadvertently connecting across samples (e.g., if coordinates from different sections overlap numerically). If necessary, build separate graphs per `Sample_ID`.

   For Step 3 (Complexity ~ entropy + UMI + Purity):
   - Do this **within each focal non-CM population** as planned; do not pool all populations for the primary test, because baseline QC differences are large.
   - Standardize predictors within each population (z-score UMI Count, Purity, entropy) to stabilize coefficients and make cross-population comparison easier.
   - Explicitly examine:
     - The coefficient and p-value for **entropy** after adjusting for UMI Count and Purity.
     - The **partial correlation** between Complexity and entropy conditional on Purity and UMI.
   - Pay special attention to fibroblasts and endothelial subsets where:
     - Purity is relatively high (e.g., aFibro, VEC, VSMC, VIC, aEndocardial) — these are the best chance for biologically meaningful high-complexity hubs.
     - Purity is low (e.g., vFibro, EPDC, BEC, vEndocardial) — the bar is higher: you’ll need to show robust entropy–complexity association within a **restricted higher-purity subset**.

   For Step 4 (defining “interface hubs”):
   - Given the strong Complexity–Purity anti-correlation, I’d suggest:
     - First, define a **purity floor** per population (e.g., exclude the bottom 20–30% of Purity within each population before defining hubs).
     - Then, among those retained cells, define hubs as:
       - Top 10% of entropy (or top 5–10% depending on how heavy-tailed entropy is).
       - Optionally confirm that their Complexity is **above the population median** but not driven by extreme low-purity outliers.
   - When you compare UMI Count and Complexity for hubs vs non-hubs, always:
     - Match or regress-out Purity (e.g., stratify by Purity bins or include Purity as covariate in a regression-based comparison alongside Mann–Whitney tests).
     - Report effect sizes along with p-values; with 200k+ cells, almost everything will be “significant,” so effect size is key.

5. **How to interpret future outcomes in light of these QC results**

   - If, after adjusting for Purity and UMI Count, neighborhood entropy remains a **strong, positive predictor** of Complexity within fibroblasts / endothelial cells, that will support the idea that **more transcriptionally complex cells sit in more heterogeneous neighborhoods**, consistent with interface/hub states.
   - If that effect is only present **at low Purity**, and disappears when restricting to higher-purity cells, then the hypothesis would be largely refuted: the “complexity”–heterogeneity relationship would be mostly technical/mixed-pixel.
   - Later, when you assess spatial localization (Step 6) relative to ventricular vs atrial cardiomyocyte territories, you’ll be able to ask a stronger question: **among cells where purity is adequate and complexity is not just a segmentation artifact, do high-entropy, high-complexity cells preferentially sit at atrial–ventricular boundaries?**

In summary, the current step successfully establishes:
- Which QC metrics are available and clean.
- The strong Complexity–Purity tension that must be controlled in all downstream modeling.
- Baseline per-population QC differences that should guide population-specific hub definitions.

The next steps you’ve planned (entropy computation, regression adjusting for UMI and Purity, hub definition with a purity floor) are exactly what is needed to turn these QC insights into a test of the hub / heterogeneity hypothesis that is not confounded by UMI depth or purity.

## Next Steps
Step 1: Compute per-cell spatial neighborhood entropy of Populations labels from k-nearest neighbors in physical space (using adata.obsm['spatial']), constructing neighborhoods within each Sample_ID to avoid cross-section mixing; store entropy scores in adata.obs for k in {10, 20, 50}, and designate spatial_entropy_k20 as the primary entropy metric while using k=10 and k=50 only for sensitivity summaries overall and by Populations.
Step 2: Within major non-cardiomyocyte Populations of interest (vFibro, aFibro, adFibro, EPDC, BEC, VEC, VSMC, Pericyte, VIC, aEndocardial, vEndocardial), model Complexity as a function of spatial_entropy_k20, Purity, and UMI Count using population-specific multiple linear regression implemented via np.linalg.lstsq and partial correlations via residualization, and for each population report whether entropy remains a significant positive predictor of Complexity after adjustment, including analyses restricted to the top 70% of Purity.
Step 3: Define 'interface hub' cells within each focal non-cardiomyocyte population by first excluding the lowest 30% of cells by Purity (per population), then selecting among the remaining cells those in the top 10% of spatial_entropy_k20; compare hub vs non-hub cells in terms of Complexity, Purity, and UMI Count using Mann–Whitney U tests and regression-based contrasts that adjust for Purity, and report effect sizes and p-values.
Step 4: Perform within-population differential expression analysis between interface hub and non-hub cells using sc.tl.rank_genes_groups (method='wilcoxon') on each focal population separately with a binary hub_status label (hub vs nonhub), summarizing for each population the top upregulated genes in hubs (names, log fold-changes, test statistics, and adjusted p-values), with particular attention to genes consistent with signaling or interaction roles given the panel.
Step 5: Assess spatial positioning of interface hub cells relative to ventricular vs atrial cardiomyocyte territories by defining ventricular CMs (Populations starting with 'vCM-') and atrial CMs (Populations starting with 'aCM-'), computing per non-cardiomyocyte cell its distances to the nearest ventricular and atrial cardiomyocyte within the same Sample_ID, deriving a boundary-proximity metric (e.g., |d_ventricular - d_atrial|), and statistically comparing this metric between hub and non-hub cells within each population using Mann–Whitney U tests to determine whether hubs are enriched near atrial–ventricular interfaces.

## Compute per-cell spatial neighborhood Shannon entropy of Populations labels from k-nearest neighbors in physical space within each Sample_ID, store these entropy scores for multiple k (with k=20 designated as the primary metric), and summarize their distributions overall and by Populations while robustly handling missing Populations labels and small-sample edge cases.

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from scipy.spatial import cKDTree

# Ensure required columns and embeddings are present
required_obs_cols = ["Populations", "Sample_ID", "UMI Count", "Complexity", "Purity"]
missing_cols = [c for c in required_obs_cols if c not in adata.obs.columns]
if missing_cols:
    print("Missing required obs columns:", missing_cols)

if "Populations" not in adata.obs.columns:
    raise ValueError("Populations column is required in adata.obs for entropy computation.")

if "spatial" not in adata.obsm.keys():
    raise ValueError("adata.obsm['spatial'] is required for spatial kNN entropy computation but is missing.")

# Work with a dense copy of spatial coordinates
coords = np.asarray(adata.obsm["spatial"])  # shape (n_cells, 2)

# Normalize/clean key obs columns (numeric casting where appropriate)
for col in ["UMI Count", "Complexity", "Purity"]:
    if col in adata.obs.columns:
        adata.obs[col] = pd.to_numeric(adata.obs[col], errors="coerce")

# Define k values for neighborhood entropy and primary k for downstream analyses
k_values = [10, 20, 50]
primary_k = 20

# Prepare containers for entropy scores
for k in k_values:
    adata.obs[f"spatial_entropy_k{k}"] = np.nan

# Helper to compute Shannon entropy from a list/array of categorical labels (natural log base: nats)
def shannon_entropy(labels):
    if len(labels) == 0:
        return np.nan
    counts = np.array(list(Counter(labels).values()), dtype=float)
    probs = counts / counts.sum()
    # Use natural log; base choice only rescales entropy (units: nats)
    return -(probs * np.log(probs)).sum()

# Determine per-cell sample assignments
if "Sample_ID" in adata.obs.columns:
    samples = adata.obs["Sample_ID"].astype(str).values
else:
    # Fallback: treat all cells as one sample
    samples = np.array(["all_one_sample"] * adata.n_obs)

unique_samples = np.unique(samples)

print(f"Computing spatial neighborhood entropy within {len(unique_samples)} sample(s)...")

for s in unique_samples:
    mask = samples == s
    idx = np.where(mask)[0]
    if idx.size == 0:
        continue

    coords_s = coords[mask, :]

    # Handle missing Populations explicitly: cells with missing Populations will receive NaN entropy
    pops_s_series = adata.obs.loc[mask, "Populations"]
    pops_s = pops_s_series.values
    has_pop = pops_s_series.notna().values

    # Build kD-tree for this sample using all cells with coordinates
    tree = cKDTree(coords_s)

    for k in k_values:
        # Query up to k+1 neighbors (including self when possible); k acts as a maximum
        effective_k = min(k + 1, coords_s.shape[0])
        if effective_k == 0:
            continue

        dists, neigh_idx = tree.query(coords_s, k=effective_k)
        # Ensure neigh_idx is 2D for consistent indexing
        neigh_idx = np.atleast_2d(neigh_idx)

        entropies = np.full(coords_s.shape[0], np.nan, dtype=float)

        for i in range(coords_s.shape[0]):
            # If this cell has missing Populations, leave entropy as NaN
            if not has_pop[i]:
                continue

            neighbors = neigh_idx[i]
            # Remove self (assumed to be at the same index) and any duplicates
            unique_neighbors = [n for n in neighbors if n != i]
            # Truncate to at most k neighbors
            if len(unique_neighbors) > k:
                unique_neighbors = unique_neighbors[:k]

            # If there are no valid neighbors (e.g., tiny sample), leave entropy as NaN
            if len(unique_neighbors) == 0:
                continue

            # Map neighbor indices to Populations labels, skipping neighbors with missing Populations
            neigh_pops = [pops_s[n] for n in unique_neighbors if has_pop[n]]

            if len(neigh_pops) == 0:
                continue

            entropies[i] = shannon_entropy(neigh_pops)

        adata.obs.loc[mask, f"spatial_entropy_k{k}"] = entropies
        print(f"  Sample {s}: computed spatial_entropy_k{k} for {coords_s.shape[0]} cells (k treated as max neighbors; some cells may have fewer due to sample size or missing Populations).")

# Summarize entropy distributions overall and by Populations
for k in k_values:
    col = f"spatial_entropy_k{k}"
    series = pd.to_numeric(adata.obs[col], errors="coerce")
    print(f"\nOverall summary for {col}:")
    print(series.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_string())

    if "Populations" in adata.obs.columns:
        tmp = pd.DataFrame({col: series, "Populations": adata.obs["Populations"].values})
        grp = tmp.groupby("Populations")[col]
        print(f"\nPer-Populations mean, std, and count for {col}:")
        per_pop_summary = grp.agg(["mean", "std", "count"]).sort_values("mean", ascending=False)
        print(per_pop_summary.to_string(float_format=lambda x: f"{x:.3f}"))

print("\nSpatial neighborhood entropy computation complete. Columns added:", ", ".join(f"spatial_entropy_k{k}" for k in k_values), f"(primary metric: spatial_entropy_k{primary_k}).")


Computing spatial neighborhood entropy within 3 sample(s)...


  Sample R77_4C4: computed spatial_entropy_k10 for 72962 cells (k treated as max neighbors; some cells may have fewer due to sample size or missing Populations).


  Sample R77_4C4: computed spatial_entropy_k20 for 72962 cells (k treated as max neighbors; some cells may have fewer due to sample size or missing Populations).


  Sample R77_4C4: computed spatial_entropy_k50 for 72962 cells (k treated as max neighbors; some cells may have fewer due to sample size or missing Populations).


  Sample R78_4C12: computed spatial_entropy_k10 for 75782 cells (k treated as max neighbors; some cells may have fewer due to sample size or missing Populations).


  Sample R78_4C12: computed spatial_entropy_k20 for 75782 cells (k treated as max neighbors; some cells may have fewer due to sample size or missing Populations).


  Sample R78_4C12: computed spatial_entropy_k50 for 75782 cells (k treated as max neighbors; some cells may have fewer due to sample size or missing Populations).


  Sample R78_4C15: computed spatial_entropy_k10 for 79891 cells (k treated as max neighbors; some cells may have fewer due to sample size or missing Populations).


  Sample R78_4C15: computed spatial_entropy_k20 for 79891 cells (k treated as max neighbors; some cells may have fewer due to sample size or missing Populations).


  Sample R78_4C15: computed spatial_entropy_k50 for 79891 cells (k treated as max neighbors; some cells may have fewer due to sample size or missing Populations).

Overall summary for spatial_entropy_k10:
count    228635.000000
mean          0.923611
std           0.444701
min          -0.000000
5%           -0.000000
25%           0.639032
50%           0.943348
75%           1.279854
95%           1.557113
max           2.163956

Per-Populations mean, std, and count for spatial_entropy_k10:
                   mean   std  count
Populations                         
Pericyte          1.231 0.308   5458
vCM-RV-Compact    1.227 0.295   9488
vCM-Proliferating 1.173 0.310  17584
vFibro            1.149 0.346  16624
vCM-RV-Trabecular 1.126 0.329   8052
BEC               1.124 0.348  12248
vCM-LV-Compact    1.112 0.323  30380
Neuronal          1.055 0.388   1027
LEC               1.054 0.360   1292
vEndocardial      1.015 0.339  10887
Epicardial        0.996 0.375   2356
WBC               0.9

### Agent Interpretation

The entropy computation step looks technically sound and provides a good foundation for the downstream tests of the hypothesis. A few key points and suggestions, focused on what they imply for the next analysis steps and interpretation:

1. **Entropy distributions and dynamic range**
   - The overall `spatial_entropy_k20` distribution has a wide spread (5th–95th percentile ~0.20–1.70, max ~2.43), so there is enough dynamic range to define “high-entropy” vs “low-entropy” neighborhoods.
   - The lower bound is effectively zero for some cells, which is interpretable as very homogeneous neighborhoods; this will be useful as a contrast group when looking for interface-like “hubs”.

2. **Interpretation of per-population entropy levels**
   - Many *ventricular cardiomyocyte* subtypes and **Pericytes/BEC/vFibro** have the highest mean entropy across all ks. This indicates these cells often sit in spatially mixed neighborhoods (multiple Populations labels).
   - In contrast, **atrial cardiomyocyte (aCM-LA/RA)** and **VICs** have the lowest entropy means, suggesting more homogeneous, “single-population–dominated” territories.
   - Among your *focal non-CM populations*:
     - `vFibro` and `EPDC` have relatively high entropy (1.32 and 1.08 at k20) → promising candidates for housing “interface hub” cells.
     - `BEC`, `VEC`, `Pericyte` also show high mean entropy, consistently across k.
     - Atrial-associated fibroblasts (`aFibro`, `adFibro`) and `VIC`, `VSMC` have lower mean entropy, so interface hubs—if present—will be a more restricted subgroup and potentially biologically interesting (cells deviating from their population’s usual homogeneous context).

   This pattern is already qualitatively consistent with the idea that certain stromal/vascular populations are enriched at mixed-type boundaries, though the hypothesis is specifically about atrial–ventricular interfaces (to be tested explicitly later).

3. **Consistency across k values (sensitivity)**
   - Rankings of populations by mean entropy are largely stable across k=10, 20, 50, with only minor shifts. This stability supports your choice of k=20 as the primary metric and suggests the entropy signal is not overly sensitive to neighborhood size.
   - For downstream robustness checks, it will be straightforward to re-run key associations using k=10 or k=50 to verify that any Complexity–entropy relationship and “hub” definition are not an artifact of k.

4. **Code / method considerations before moving on**
   - The implementation respects within-sample neighborhoods and handles missing Populations by assigning NaN entropy, which is appropriate.
   - Entropy is computed on *neighbors only* (excluding self); this aligns with the conceptual definition of “local neighborhood heterogeneity”.
   - One caveat: entropy is computed over *cell-type labels*, not anatomical labels. High entropy here means “diverse mixtures of labeled Populations”, which may reflect:
     - true tissue interfaces (e.g., fibroblast–endothelial–CM boundaries),
     - small intermingled microvascular/stromal niches inside a larger atrial or ventricular domain,
     - or local label heterogeneity even within a single gross anatomical region.
     
     The later step where you compare distances to ventricular vs atrial CMs will be critical to tie this cell-type-level heterogeneity specifically to atrial–ventricular boundaries, rather than just “any mixed neighborhood”.

5. **Immediate next analyses to prioritize**
   For the stated hypothesis, the following are the most informative next steps:

   **a. Complexity–entropy–Purity–UMI modeling (planned step 2)**
   - Run the per-population multiple linear regression:
     - Outcome: `Complexity`
     - Predictors: `spatial_entropy_k20`, `Purity`, `UMI Count`
   - Focus on the key non-CM populations: `vFibro, aFibro, adFibro, EPDC, BEC, VEC, VSMC, Pericyte, VIC, aEndocardial, vEndocardial`.
   - Extract:
     - The coefficient and p-value for `spatial_entropy_k20` in each population.
     - Partial correlations (either by residualizing Complexity on Purity+UMI and entropy on Purity+UMI, or from the regression).
   - Repeat restricted to the top 70% Purity per population (as in your plan) to ensure the signal is not driven by doublets/multiplets.
   - If the hypothesis is correct, you should see a **positive** and **significant** association between entropy and Complexity in at least a subset of these populations, especially those that already show high mean entropy (vFibro, EPDC, BEC, VEC, Pericyte).

   This will directly address the first part of the hypothesis (“Complexity is positively associated with spatial entropy after adjustment”).

   **b. “Interface hub” definition and characterization (planned step 3)**
   - Using the new `spatial_entropy_k20`:
     - Filter out the lowest 30% of cells by Purity in each focal population.
     - Among the remaining, define hubs as top 10% `spatial_entropy_k20`.
   - Given the population-level entropy means, the **absolute entropy thresholds** that define hubs will vary strongly between populations:
     - In low-entropy populations (e.g., `VIC`, `aCM-RA/LA`, `VSMC`), hub cells will be those *relatively* high within that population, which might highlight rare interface-like cells in otherwise homogeneous tissue.
     - In high-entropy populations (e.g., `vFibro`, `BEC`), hubs will be the extreme “super-mixed” neighborhoods; these may be particularly informative for interface biology.
   - Compare hubs vs non-hubs (within each population) for:
     - Complexity (Mann–Whitney and regression adjusting for Purity & UMI).
     - Purity (to ensure they are not systematically lower even after your 30% cutoff).
   - A pattern consistent with the hypothesis would be:
     - Hubs have **higher Complexity** than non-hubs even after adjusting for Purity/UMI.
     - Hubs are not just the lowest-purity cells (ideally similar or slightly lower Purity, but not dramatically so).

6. **Analytical refinements / sanity checks to consider**
   - Before regression, inspect simple scatterplots of Complexity vs `spatial_entropy_k20` for a few key populations (e.g., vFibro, aFibro, EPDC, VEC), color by Purity or UMI. This can reveal:
     - Obvious non-linearities (e.g., plateau at high entropy).
     - Outliers with extremely high entropy that might warrant filtering or robustness checks.
   - Evaluate whether entropy correlates strongly with Purity or UMI themselves; if so, the interpretation of the Complexity–entropy association will need to be careful. Strong collinearity could inflate variances in the regression.

7. **Linking entropy to atrial–ventricular interfaces (planned step 5)**
   - Because ventricular CMs have notably higher entropy than atrial CMs, there is already a hint that the ventricular territories are more mixed in terms of label composition. However, this by itself doesn’t prove an atrial–ventricular interface phenomenon.
   - When you compute distance-based boundary metrics (`|d_ventricular - d_atrial|`), you will be able to test explicitly:
     - Within each non-CM population, do high-entropy hubs lie closer to the putative AV boundary (i.e., smaller |d_v − d_a|) than non-hubs?
   - If you observe:
     - Complexity-high, high-entropy hubs with **preferential enrichment near low |d_v − d_a|**, especially in vFibro/aFibro, EPDC, endothelial populations, that would align well with the full hypothesis (complex, heterogeneous “interface” cells at AV boundaries).
   - Here, doing sensitivity analyses with `spatial_entropy_k10` and `k50` might help verify that boundary enrichment is robust to neighborhood definition.

8. **Distinctness from the paper and prior analyses**
   - The current entropy analysis is fundamentally about **local cell-type diversity** in physical space, whereas your prior analyses were centered on:
     - Maturation scores and their spatial autocorrelation.
     - TF/signaling scores (failed technically).
   - The per-population entropy profiles, hub definitions, and especially the AV-boundary distance metric are conceptually distinct from those earlier efforts and should remain distinct from typical “who co-localizes with whom” community analyses in the original paper, as long as you focus on:
     - Complexity and Purity-adjusted associations.
     - Explicit AV-boundary metrics and “interface hubs” as defined via entropy.

In summary, the entropy metrics look well-behaved and informative, with clear differences among populations that will be useful for defining and interpreting “interface hubs.” The next critical step is the population-specific regression of Complexity on entropy (with Purity/UMI adjustment), followed by hub vs non-hub contrasts and the spatial AV-boundary analysis to test whether these high-entropy, high-Complexity cells are indeed enriched at atrial–ventricular boundaries.

## Next Steps
Step 1: Within major non-cardiomyocyte populations of interest (vFibro, aFibro, adFibro, EPDC, BEC, VEC, VSMC, Pericyte, VIC, aEndocardial, vEndocardial), model Complexity as a linear function of spatial_entropy_k20, Purity, and UMI Count using population-specific multiple linear regression (via np.linalg.lstsq) and partial correlation (via residualization), and for each population report whether spatial_entropy_k20 is a significant positive predictor of Complexity after adjustment, including analyses restricted to the top 70% of Purity per population (based on per-population 0.3 Purity quantiles).
Step 2: Define 'interface hub' cells within each focal non-cardiomyocyte population by first excluding the lowest 30% of cells by Purity (per population), then selecting among the remaining cells those in the top 10% of spatial_entropy_k20; compare hub vs non-hub cells in terms of Complexity, Purity, and UMI Count using Mann–Whitney U tests and regression-based contrasts that adjust for Purity and UMI Count, and report effect sizes and p-values.
Step 3: Perform within-population differential expression analysis between interface hub and non-hub cells using sc.tl.rank_genes_groups (method='wilcoxon') on each focal population separately with a binary hub_status label (hub vs nonhub), and summarize for each population the top upregulated genes in hubs (names, log fold-changes, test statistics, and adjusted p-values) to identify potential signaling or interaction-related markers of interface states.
Step 4: Assess spatial positioning of interface hub cells relative to ventricular vs atrial cardiomyocyte territories by defining ventricular CMs (Populations starting with 'vCM-') and atrial CMs (Populations starting with 'aCM-'), computing for each non-cardiomyocyte cell its distances to the nearest ventricular and atrial cardiomyocyte within the same Sample_ID, deriving a boundary-proximity metric such as |d_ventricular - d_atrial|, and statistically comparing this metric between hub and non-hub cells within each population using Mann–Whitney U tests to determine whether hubs are enriched near putative atrial–ventricular interfaces.

## Fit, within each focal non-cardiomyocyte population, a linear multiple regression of Complexity on standardized spatial entropy, Purity, and UMI Count using np.linalg.lstsq, and compute partial correlations of Complexity with entropy after regressing out Purity and UMI. This tests whether spatial entropy remains a significant linear predictor of Complexity after accounting for Purity and UMI, both in all cells and in a per-population top-70% Purity subset.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Define focal non-cardiomyocyte populations for analysis
focal_pops = [
    'vFibro', 'aFibro', 'adFibro', 'EPDC', 'BEC', 'VEC',
    'VSMC', 'Pericyte', 'VIC', 'aEndocardial', 'vEndocardial'
]

# Check required columns
required_cols = ['Complexity', 'Purity', 'UMI Count', 'Populations', 'spatial_entropy_k20']
missing = [c for c in required_cols if c not in adata.obs.columns]
if missing:
    raise ValueError(f"Missing required columns in adata.obs: {missing}")

# Ensure numeric types where needed
for col in ['Complexity', 'Purity', 'UMI Count', 'spatial_entropy_k20']:
    adata.obs[col] = pd.to_numeric(adata.obs[col], errors='coerce')

print("Per-population linear regression of Complexity on entropy, Purity, and UMI Count")
print("(full data and restricted to top 70% Purity per population; linear effects only)\n")

results_rows = []

# Helper to fit regression and partial correlation in a given DataFrame
def fit_and_summarize(df_sub, pop, subset_label):
    # Drop rows with any missing values in predictors or outcome
    df_sub = df_sub.dropna(subset=['Complexity', 'Purity', 'UMI Count', 'spatial_entropy_k20'])
    n = df_sub.shape[0]
    if n < 50:
        print(f"{pop} ({subset_label}): fewer than 50 cells with complete data; skipping.")
        return

    y = df_sub['Complexity'].values.astype(float)
    X = df_sub[['spatial_entropy_k20', 'Purity', 'UMI Count']].values.astype(float)

    # Standardize predictors for comparability and numerical stability
    X_mean = X.mean(axis=0)
    X_std = X.std(axis=0, ddof=0)
    X_std[X_std == 0] = 1.0
    X_z = (X - X_mean) / X_std

    # Add intercept
    X_design = np.column_stack([np.ones(X_z.shape[0]), X_z])

    # Fit via least squares (linear model; non-linearities are not modeled here)
    beta, residuals, rank, s = np.linalg.lstsq(X_design, y, rcond=None)

    # Predicted and residuals for sigma^2
    y_hat = X_design @ beta
    resid = y - y_hat
    dof = X_design.shape[0] - X_design.shape[1]
    if dof > 0:
        sigma2 = (resid ** 2).sum() / dof
    else:
        sigma2 = np.nan

    # Variance-covariance matrix of betas
    XtX = X_design.T @ X_design
    try:
        XtX_inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        XtX_inv = np.linalg.pinv(XtX)
    se_beta = np.sqrt(np.diag(XtX_inv) * sigma2)

    # t-stats and p-values for coefficients (2-sided); guard against zero SE
    with np.errstate(divide='ignore', invalid='ignore'):
        t_stats = beta / se_beta
    if dof > 0:
        p_vals = 2 * stats.t.sf(np.abs(t_stats), df=dof)
    else:
        p_vals = np.full_like(t_stats, np.nan)

    # Index mapping: intercept=0, entropy=1, purity=2, UMI=3
    idx_entropy = 1
    idx_purity = 2
    idx_umi = 3

    # Partial correlation between Complexity and entropy adjusted for Purity and UMI
    Z = df_sub[['Purity', 'UMI Count']].values.astype(float)
    Z_design = np.column_stack([np.ones(Z.shape[0]), Z])

    # Residualize y
    beta_y, _, _, _ = np.linalg.lstsq(Z_design, y, rcond=None)
    y_resid = y - Z_design @ beta_y

    # Residualize entropy
    entropy = df_sub['spatial_entropy_k20'].values.astype(float)
    beta_e, _, _, _ = np.linalg.lstsq(Z_design, entropy, rcond=None)
    e_resid = entropy - Z_design @ beta_e

    if np.std(y_resid) > 0 and np.std(e_resid) > 0:
        r_partial, p_partial = stats.pearsonr(y_resid, e_resid)
    else:
        r_partial, p_partial = np.nan, np.nan

    # Store numeric results
    results_rows.append({
        'Population': pop,
        'Subset': subset_label,
        'N_cells': n,
        'coef_entropy_std': beta[idx_entropy],
        'se_entropy_std': se_beta[idx_entropy],
        't_entropy_std': t_stats[idx_entropy],
        'p_entropy_std': p_vals[idx_entropy],
        'coef_purity_std': beta[idx_purity],
        'p_purity_std': p_vals[idx_purity],
        'coef_umi_std': beta[idx_umi],
        'p_umi_std': p_vals[idx_umi],
        'partial_r_entropy': r_partial,
        'partial_p_entropy': p_partial,
        'rank_X_design': rank
    })

    print(f"{pop} ({subset_label}): N={n}, coef_entropy_std={beta[idx_entropy]:.3f}, "
          f"p_entropy_std={p_vals[idx_entropy]:.3e}, partial_r_entropy={r_partial:.3f}, "
          f"partial_p_entropy={p_partial:.3e}, rank={rank}")


for pop in focal_pops:
    mask_pop = adata.obs['Populations'] == pop
    n_pop = mask_pop.sum()
    if n_pop < 50:
        print(f"Skipping {pop}: only {n_pop} cells (require >= 50).")
        continue

    df = adata.obs.loc[mask_pop, ['Complexity', 'Purity', 'UMI Count', 'spatial_entropy_k20']]

    # Fit on all available cells in this population
    fit_and_summarize(df.copy(), pop=pop, subset_label='all')

    # Restrict to top 70% Purity in this population (per-population 0.3 quantile threshold)
    purity_threshold = df['Purity'].quantile(0.3)
    df_high_purity = df.loc[df['Purity'] >= purity_threshold]
    if df_high_purity.shape[0] >= 30:
        fit_and_summarize(df_high_purity.copy(), pop=pop, subset_label='top70_purity')
    else:
        print(f"{pop}: top70_purity subset has only {df_high_purity.shape[0]} cells; skipping restricted analysis.")

# Aggregate results table for reference
if results_rows:
    results_df = pd.DataFrame(results_rows)
    results_df = results_df.sort_values(['Population', 'Subset'])
    print("\nSummary table of entropy effects by population and subset (standardized predictors; linear model):")
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
else:
    print("No valid populations had sufficient cells for regression analysis.")


Per-population linear regression of Complexity on entropy, Purity, and UMI Count
(full data and restricted to top 70% Purity per population; linear effects only)

vFibro (all): N=16624, coef_entropy_std=0.003, p_entropy_std=8.484e-01, partial_r_entropy=0.001, partial_p_entropy=8.484e-01, rank=4
vFibro (top70_purity): N=11637, coef_entropy_std=0.018, p_entropy_std=3.739e-01, partial_r_entropy=0.008, partial_p_entropy=3.738e-01, rank=4
aFibro (all): N=7417, coef_entropy_std=0.178, p_entropy_std=2.165e-08, partial_r_entropy=0.065, partial_p_entropy=2.156e-08, rank=4
aFibro (top70_purity): N=5192, coef_entropy_std=0.173, p_entropy_std=3.416e-08, partial_r_entropy=0.077, partial_p_entropy=3.395e-08, rank=4
adFibro (all): N=1562, coef_entropy_std=0.401, p_entropy_std=1.597e-13, partial_r_entropy=0.185, partial_p_entropy=1.541e-13, rank=4
adFibro (top70_purity): N=1093, coef_entropy_std=0.168, p_entropy_std=1.562e-03, partial_r_entropy=0.096, partial_p_entropy=1.547e-03, rank=4
EPDC (all): N=

### Agent Interpretation

These regressions already give a fairly clear answer about the core hypothesis and point directly to which lineages are worth pursuing in the “interface hub” analysis versus which ones may not yield interesting biology.

**1. Hypothesis evaluation (Complexity vs spatial_entropy_k20 after adjusting for Purity and UMI)**

You tested the key piece of the hypothesis: within each non-CM population, does spatial entropy predict Complexity after controlling for Purity and UMI-Count?

Summary by population (focusing on entropy effects on standardized scale and partial r):

- **Clear positive associations (supportive of hypothesis):**
  - **aFibro**: coef ≈ 0.18, partial r ≈ 0.065–0.077, p ≈ 10⁻⁸ in both all and top70% Purity.
  - **adFibro**: coef ≈ 0.40 (all) / 0.17 (top70), partial r ≈ 0.19 / 0.096, p ≪ 0.01.
  - **BEC**: coef ≈ 0.15–0.16, partial r ≈ 0.06–0.07, p ≈ 10⁻¹¹–10⁻¹⁰.
  - **VEC**: coef ≈ 0.40–0.49, partial r ≈ 0.14–0.18, p ≈ 10⁻¹⁸–10⁻²¹.
  - **aEndocardial**: coef ≈ 0.42 / 0.25, partial r ≈ 0.20 / 0.15, p ≈ 10⁻⁴¹–10⁻¹⁷.
  - **vEndocardial**: coef ≈ 0.27–0.33, partial r ≈ 0.13–0.17, p ≈ 10⁻⁴⁴–10⁻⁴⁷.
  - **VIC (all only)**: small but significant positive effect (coef ≈ 0.13, partial r ≈ 0.037), which disappears in top70% Purity.

  These lineages fit the hypothesized pattern: **higher local spatial heterogeneity is associated with higher Complexity independent of Purity and UMIs**. Effect sizes are modest (as expected for single-cell complexity metrics) but robust and consistent across full and high-purity subsets for most of these types.

- **Negative or absent associations (do *not* support the hypothesis as stated):**
  - **EPDC**: strong and **negative** association (coef ≈ -0.83 / -0.95, partial r ≈ -0.30 / -0.33, p ~ 10⁻¹⁷¹). This is a clear counterexample: higher-entropy EPDC cells are less complex after adjustment.
  - **VSMC**: no signal in “all” (coef ≈ 0.028, p ≈ 0.59) but significantly **negative** in top70% Purity (coef ≈ -0.20, partial r ≈ -0.062, p ≈ 4×10⁻⁴).
  - **vFibro**: essentially no relationship (coef ≈ 0.00–0.02, partial r ≈ 0.001–0.008, p ≫ 0.05).
  - **Pericyte**: weak, non-significant positive trends (coef ≈ 0.03–0.04; partial r ~0.014–0.020; p ~0.2–0.3), so no compelling effect.
  - **VIC (top70% Purity)**: entropy effect disappears in high-purity subset.

**Interpretation relative to the hypothesis:**

- The **hypothesis is partially validated**:
  - Strong support in **atrial/AV fibroblasts (aFibro, adFibro)**, **vascular/valve endothelia (BEC, VEC)**, and **atrial/ventricular endocardium (aEndocardial, vEndocardial)**, where high-entropy cells are genuinely more complex even when controlling for Purity and UMI Count and after excluding low-purity cells.
  - These are prime candidates for biologically distinct “interface/high-entropy, high-complexity” states.

- But it is **not universally true across all non-CM lineages**:
  - **EPDC** and **VSMC** show the opposite trend in high-purity cells (higher entropy → *lower* Complexity).
  - **vFibro, Pericyte, and high-purity VIC** do not show a meaningful association.

So, for the subsequent steps, it will be important to treat lineages differently rather than assuming a universal positive relationship.

---

**2. Guidance for defining and analyzing “interface hubs” (next steps of your plan)**

Given these results, here is how I’d prioritize and interpret the upcoming “hub” analyses:

1. **Focus interface hub analyses on lineages where entropy–Complexity association is robustly positive:**
   - **High-priority populations for hub vs non-hub comparisons and DE:**
     - aFibro, adFibro
     - BEC, VEC
     - aEndocardial, vEndocardial
   - **Moderate-priority / exploratory:**
     - VIC (but be cautious: signal is driven by lower-purity strata; high-purity VIC shows no entropy effect).

   For these types, you can interpret “interface hubs” (top 10% entropy among top 70% purity) as **enriched for high-Complexity transcriptional states**, not just technical mixtures.

2. **Use high-purity subset for hub definition as planned.**
   - Your regression shows the entropy–Complexity relationships largely **persist or strengthen** in the top 70% purity subset for the positive-effect lineages, which justifies using that subset to define hubs.
   - This is crucial to support the “not purely low-purity artifacts” part of the hypothesis.

3. **Adjust for Purity and UMI in hub vs non-hub contrasts.**
   - In most populations, **Purity and UMI have strong negative coefficients** in the regressions (i.e., lower purity and sometimes lower UMI = higher Complexity). That’s a potential confound.
   - When you compare hubs vs non-hubs:
     - Do both **Mann–Whitney U tests on raw Complexity** and
     - **Regression contrasts** (Complexity ~ hub_status + Purity + UMI) within each population.
   - For the positive-effect lineages, you should see hub_status remain significant with a positive coefficient, further consolidating the idea of biologically meaningful states.

4. **Interpretation for EPDC and VSMC in hub analysis:**
   - **EPDC**: since higher entropy is strongly associated with *lower* Complexity even in high-purity cells, your “interface hubs” (high entropy, high purity) will likely be **lower-complexity EPDC states**. This is biologically interesting but opposite to the original hypothesis.
     - Still worth including, but interpret hubs as **distinct low-complexity interface-like EPDC states**, potentially reflecting a more restricted or specialized epigenetic program at interfaces.
   - **VSMC**: similarly, in top70% purity, high entropy is linked to lower Complexity. For VSMC hubs, expect **reduced Complexity relative to non-hubs after adjustment**.
   - For both, you can explicitly frame them as “inverse-relationship interface states” and contrast their DE patterns with those from the positively associated lineages; it gives you a nuanced story rather than a single monotonic pattern.

5. **Deprioritize vFibro and Pericytes for the interface-Complexity story.**
   - The near-zero effect sizes and non-significant partial correlations suggest that for **vFibro and Pericytes**, high spatial entropy does not map cleanly onto Complexity after adjustment.
   - You could still define hubs for completeness and for the spatial positioning analysis, but I would:
     - Expect smaller or absent differences in Complexity between hub and non-hub cells.
     - Put them lower priority for extensive DE and interpretation.

---

**3. Suggestions for the DE step (hub vs non-hub within populations)**

Given that Complexity is only weakly correlated with entropy (partial r ~0.06–0.18 even in the strongest cases), hubs are not just “top complexity outliers”; they’re **spatially-defined subgroups**. The DE step is where you can extract biological meaning:

- For **positive-effect lineages (aFibro, adFibro, BEC, VEC, aEndocardial, vEndocardial)**:
  - DE genes up in hubs will likely reflect **signaling, interaction, and plasticity** states at spatially heterogeneous interfaces.
  - Because multiple populations show similar entropy–Complexity behavior, look for:
    - Recurrent genes upregulated in hubs across several lineages (e.g., shared signaling ligands/receptors).
    - Population-specific interface programs (e.g., fibroblast vs endocardial interface signatures).

- For **EPDC and VSMC (negative association)**:
  - DE genes up in hubs will instead mark **low-complexity, high-entropy states**.
  - Compare these signatures to positive-effect lineages:
    - Are there distinct sets of developmental regulators or extracellular matrix genes that define EPDC/VSMC “simplification” at interfaces?

---

**4. Spatial positioning analysis (later step: distances to atrial vs ventricular CMs)**

The regression results suggest that **endocardial and endothelial lineages** are particularly promising for the spatial boundary analysis:

- **aEndocardial, vEndocardial, VEC, BEC**:
  - Strong positive entropy–Complexity relationships.
  - Biologically plausible to act as interface layers between atrial and ventricular territories or across vascular/valve boundaries.
  - When you compute the boundary-proximity metric |d_ventricular - d_atrial|, I would especially focus on:
    - Whether endocardial and valve endothelial hubs **cluster near low boundary distance** (i.e., near atrial–ventricular interfaces).
    - Whether this is more pronounced in the positively associated populations than in EPDC/VSMC.

- **aFibro/adFibro**:
  - Also strong positive relationships; look for whether fibroblast hubs are enriched around atrial–ventricular transition zones or around specialized structures (e.g., valve/AVC regions, depending on sample metadata).

- **EPDC and VSMC**:
  - For the negative-association lineages, the spatial test can help you decide whether their high-entropy/low-complexity hubs are:
    - Also enriched near boundaries (suggesting a different kind of interface adaptation), or
    - Distributed elsewhere (which would argue that for these lineages, entropy reflects something other than a classic AV interface).

---

**5. Methodological checks / extensions to consider (if you iterate)**

- **Non-linearity and interactions:** The current model is strictly linear. For some populations (e.g., EPDC), a strong negative coefficient might partly reflect non-linear effects or interactions between entropy and Purity. If you want to explore further:
  - Fit models with quadratic terms or entropy × Purity interactions in a subset of populations.
  - Check whether the sign of the entropy effect is robust to such extensions.

- **Effect size visualization:** For interpretability and to guide the DE:
  - Plot Complexity vs spatial_entropy_k20 for a few representative populations, color-coded by Purity or UMI, with regression lines for binned Purity strata.
  - This may help visually confirm that the positive and negative trends reflect real biology and not just leverage from a small subset of cells.

---

**Bottom line for next steps:**

- **Strongly proceed** with hub definition and hub vs non-hub analyses in:
  - aFibro, adFibro, BEC, VEC, aEndocardial, vEndocardial (and optionally VIC).
- **Include EPDC and VSMC** as contrasting cases with negative entropy–Complexity coupling; interpret their hubs accordingly.
- **Treat vFibro and Pericytes as lower priority** for the “high-entropy, high-complexity” hypothesis, but you may still analyze them for completeness or unexpected spatial patterns.
- Use the upcoming DE and spatial positioning steps to articulate:
  - Which lineages host **high-entropy, high-complexity interface states**, and
  - Which lineages show **high-entropy, low-complexity** or **no clear relationship**, giving a more nuanced, lineage-specific view than the original hypothesis implied.

## Next Steps
Step 1: Use existing per-population regressions of Complexity on spatial_entropy_k20, Purity, and UMI Count (including high-purity subsets) to classify each focal lineage by the sign and significance of its entropy–Complexity coupling (positive, negative, or null), and treat these regression summaries as the formal test of whether entropy remains a predictor of Complexity after adjustment.
Step 2: Within major non-cardiomyocyte Populations (vFibro, aFibro, adFibro, EPDC, BEC, VEC, VSMC, Pericyte, VIC, aEndocardial, vEndocardial), compare 'interface hub' vs non-hub cells after excluding the lowest 30% of cells by Purity per population and defining hubs as the top 10% of spatial_entropy_k20 within the retained cells; for each population, test differences in Complexity, Purity, and UMI Count using Mann–Whitney U tests and regression-based contrasts (Complexity ~ hub_status + Purity + UMI Count), then summarize whether the observed hub_status effects on Complexity align with the previously estimated entropy–Complexity regression sign (including inverse patterns in EPDC and VSMC).
Step 3: Perform within-population differential expression analysis between interface hub and non-hub cells for each focal population using sc.tl.rank_genes_groups (method='wilcoxon') on a binary hub_status label, and for each lineage summarize the top upregulated hub markers (names, log fold-changes, statistics, and adjusted p-values), highlighting whether lineages with positive versus negative entropy–Complexity coupling show distinct categories of DE genes (e.g., signaling/interaction versus more restricted programs).

## This code defines per-population 'interface hub' cells (top 10% spatial entropy within the top 70% of Purity) among focal non-cardiomyocyte lineages and compares hub versus non-hub cells on Complexity, Purity, and UMI Count using Mann–Whitney U tests and a regression of Complexity on hub_status, Purity, and UMI Count, providing effect sizes and p-values for lineage-specific hub effects on Complexity.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Focal non-cardiomyocyte populations (same as in the regression step)
focal_pops = [
    'vFibro', 'aFibro', 'adFibro', 'EPDC', 'BEC', 'VEC',
    'VSMC', 'Pericyte', 'VIC', 'aEndocardial', 'vEndocardial'
]

required_cols = ['Complexity', 'Purity', 'UMI Count', 'Populations', 'spatial_entropy_k20']
missing = [c for c in required_cols if c not in adata.obs.columns]
if missing:
    raise ValueError(f"Missing required columns in adata.obs: {missing}")

# Ensure numeric types
for col in ['Complexity', 'Purity', 'UMI Count', 'spatial_entropy_k20']:
    adata.obs[col] = pd.to_numeric(adata.obs[col], errors='coerce')

results = []

print("Defining interface hubs (top 10% entropy within top 70% Purity) per population and comparing Complexity, Purity, and UMI Count between hub and non-hub cells.\n")

for pop in focal_pops:
    mask_pop = adata.obs['Populations'] == pop
    n_pop = int(mask_pop.sum())
    if n_pop < 50:
        print(f"Skipping {pop}: only {n_pop} cells (require >= 50).")
        continue

    df = adata.obs.loc[mask_pop, ['Complexity', 'Purity', 'UMI Count', 'spatial_entropy_k20']].copy()
    df = df.dropna(subset=['Complexity', 'Purity', 'UMI Count', 'spatial_entropy_k20'])
    if df.shape[0] < 50:
        print(f"Skipping {pop}: fewer than 50 cells with complete data after dropping NAs.")
        continue

    # Per-population purity floor: exclude bottom 30% by Purity
    purity_floor = df['Purity'].quantile(0.3)
    df_hp = df[df['Purity'] >= purity_floor].copy()
    n_hp = df_hp.shape[0]
    if n_hp < 40:
        print(f"{pop}: fewer than 40 cells after applying 30% Purity floor (N={n_hp}); skipping hub definition.")
        continue

    # Define hubs as top 10% of spatial_entropy_k20 within high-purity cells
    entropy_thresh = df_hp['spatial_entropy_k20'].quantile(0.9)
    df_hp['hub_status'] = (df_hp['spatial_entropy_k20'] >= entropy_thresh).astype(int)

    n_hub = int(df_hp['hub_status'].sum())
    n_nonhub = int((df_hp['hub_status'] == 0).sum())
    if n_hub < 10 or n_nonhub < 20:
        print(f"{pop}: insufficient hub/non-hub cells after thresholds (hubs={n_hub}, non-hubs={n_nonhub}); skipping.")
        continue

    # Mann–Whitney U tests for Complexity, Purity, UMI Count
    summary = {"Population": pop, "N_total": int(df.shape[0]), "N_hp": n_hp,
               "N_hub": n_hub, "N_nonhub": n_nonhub,
               "purity_floor": float(purity_floor), "entropy_hub_thresh": float(entropy_thresh)}

    for var in ['Complexity', 'Purity', 'UMI Count']:
        vals_hub = df_hp.loc[df_hp['hub_status'] == 1, var].values
        vals_non = df_hp.loc[df_hp['hub_status'] == 0, var].values
        # Guard against completely constant distributions in both groups
        if (len(vals_hub) == 0 or len(vals_non) == 0 or
            (np.all(vals_hub == vals_hub[0]) and np.all(vals_non == vals_non[0]))):
            u_stat, p_u, r_rb = np.nan, 1.0, 0.0
        else:
            u_stat, p_u = stats.mannwhitneyu(vals_hub, vals_non, alternative='two-sided')
            n1 = len(vals_hub)
            n2 = len(vals_non)
            # Rank-biserial effect size (positive means hubs tend to have higher values)
            r_rb = 1.0 - (2.0 * u_stat) / (n1 * n2)

        mean_hub = float(np.mean(vals_hub))
        mean_non = float(np.mean(vals_non))
        med_hub = float(np.median(vals_hub))
        med_non = float(np.median(vals_non))

        summary[f'{var}_mean_hub'] = mean_hub
        summary[f'{var}_mean_nonhub'] = mean_non
        summary[f'{var}_median_hub'] = med_hub
        summary[f'{var}_median_nonhub'] = med_non
        summary[f'{var}_U_stat'] = float(u_stat) if not np.isnan(u_stat) else np.nan
        summary[f'{var}_U_p'] = float(p_u)
        summary[f'{var}_rank_biserial'] = float(r_rb)

    # Regression-based contrast for Complexity: Complexity ~ hub_status + Purity + UMI Count
    reg_df = df_hp[['Complexity', 'Purity', 'UMI Count', 'hub_status']].dropna()
    y = reg_df['Complexity'].values.astype(float)
    X_cov = reg_df[['hub_status', 'Purity', 'UMI Count']].values.astype(float)

    # Standardize continuous covariates (Purity, UMI Count), keep hub_status as 0/1
    X_std = X_cov.copy()
    # column 0: hub_status, 1: Purity, 2: UMI
    for j in [1, 2]:
        col_mean = X_std[:, j].mean()
        col_std = X_std[:, j].std(ddof=0)
        if col_std == 0:
            col_std = 1.0
        X_std[:, j] = (X_std[:, j] - col_mean) / col_std

    X_design = np.column_stack([np.ones(X_std.shape[0]), X_std])
    beta, residuals, rank, s = np.linalg.lstsq(X_design, y, rcond=None)
    dof = X_design.shape[0] - X_design.shape[1]
    if dof > 0:
        sigma2 = float(((y - X_design @ beta) ** 2).sum() / dof)
    else:
        sigma2 = np.nan

    XtX = X_design.T @ X_design
    try:
        XtX_inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        XtX_inv = np.linalg.pinv(XtX)
    se_beta = np.sqrt(np.diag(XtX_inv) * sigma2)

    with np.errstate(divide='ignore', invalid='ignore'):
        t_stats = beta / se_beta
    if dof > 0:
        p_vals = 2 * stats.t.sf(np.abs(t_stats), df=dof)
    else:
        p_vals = np.full_like(t_stats, np.nan)

    # hub_status coefficient is at index 1 (after intercept)
    summary['Complexity_coef_hubstatus'] = float(beta[1])
    summary['Complexity_se_hubstatus'] = float(se_beta[1]) if not np.isnan(se_beta[1]) else np.nan
    summary['Complexity_t_hubstatus'] = float(t_stats[1]) if not np.isnan(t_stats[1]) else np.nan
    summary['Complexity_p_hubstatus'] = float(p_vals[1]) if not np.isnan(p_vals[1]) else np.nan
    summary['regression_dof'] = int(dof)

    results.append(summary)

    print(f"{pop}: N_hp={n_hp}, hubs={n_hub}, non-hubs={n_nonhub}; "
          f"Complexity (MWU) rank-biserial={summary['Complexity_rank_biserial']:.3f}, p={summary['Complexity_U_p']:.3e}; "
          f"regression hub_status coef={summary['Complexity_coef_hubstatus']:.3f}, p={summary['Complexity_p_hubstatus']:.3e}")

# Convert to DataFrame and print a compact summary table
if results:
    res_df = pd.DataFrame(results)
    cols_show = [
        'Population', 'N_total', 'N_hp', 'N_hub', 'N_nonhub',
        'purity_floor', 'entropy_hub_thresh',
        'Complexity_mean_hub', 'Complexity_mean_nonhub',
        'Complexity_rank_biserial', 'Complexity_U_p',
        'Complexity_coef_hubstatus', 'Complexity_p_hubstatus'
    ]
    cols_show = [c for c in cols_show if c in res_df.columns]
    print("\nSummary of hub vs non-hub Complexity differences (per population):")
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        print(res_df[cols_show].sort_values('Population').to_string(index=False, float_format=lambda x: f"{x:.4f}"))
else:
    print("No populations yielded valid hub vs non-hub comparisons.")


Defining interface hubs (top 10% entropy within top 70% Purity) per population and comparing Complexity, Purity, and UMI Count between hub and non-hub cells.

vFibro: N_hp=11637, hubs=1187, non-hubs=10450; Complexity (MWU) rank-biserial=-0.236, p=1.806e-41; regression hub_status coef=0.536, p=2.908e-19
aFibro: N_hp=5192, hubs=522, non-hubs=4670; Complexity (MWU) rank-biserial=-0.545, p=3.187e-94; regression hub_status coef=1.186, p=4.991e-33
adFibro: N_hp=1093, hubs=111, non-hubs=982; Complexity (MWU) rank-biserial=-0.373, p=5.032e-11; regression hub_status coef=1.027, p=1.067e-10
EPDC: N_hp=5980, hubs=599, non-hubs=5381; Complexity (MWU) rank-biserial=0.219, p=9.017e-19; regression hub_status coef=-0.915, p=9.041e-17
BEC: N_hp=8573, hubs=867, non-hubs=7706; Complexity (MWU) rank-biserial=-0.290, p=1.154e-45; regression hub_status coef=0.553, p=1.255e-13
VEC: N_hp=2608, hubs=271, non-hubs=2337; Complexity (MWU) rank-biserial=-0.433, p=6.631e-32; regression hub_status coef=1.075, p=3.39

### Agent Interpretation

These results are highly informative and largely supportive of the hypothesis that, within non-cardiomyocyte lineages, spatial neighborhood entropy remains a lineage-specific predictor of Complexity after adjusting for Purity and UMI, and that “interface hub” cells differ systematically from non-hubs.

Key points and how they inform next steps:

1. **Hub vs non-hub Complexity effects are large and robust in almost all populations**

   - All lineages except VIC show **strong, highly significant hub_status coefficients** in the regression `Complexity ~ hub_status + Purity + UMI Count`:
     - Positive hub_status effects on Complexity: vFibro, aFibro, adFibro, BEC, VEC, Pericyte, aEndocardial, vEndocardial (β ≈ 0.55–1.19).
     - Negative hub_status effects on Complexity: EPDC (β ≈ −0.91), VSMC (β ≈ −0.74).
     - VIC: weak/non-significant (β ≈ 0.15, p ≈ 0.10).
   - This strongly supports **lineage-specific entropy–Complexity coupling** after adjusting for Purity and UMI, and provides **clear evidence of inverse patterns specifically in EPDC and VSMC**, exactly as hypothesized.

   Interpretation:
   - For most non-CM lineages, high-entropy “interface hubs” are **more complex transcriptionally**, even given similar Purity and UMI.
   - In EPDC and VSMC, the opposite holds: high-entropy hubs are **less complex** once Purity/UMI are controlled, matching the idea of distinct, possibly more specialized or constrained interaction states at high-entropy interfaces.

   This regression-based contrast is already a good **formal test** for the hypothesis that entropy remains predictive of Complexity beyond technical covariates.

2. **Raw vs adjusted comparisons highlight the importance of covariate adjustment**

   - For most lineages, the Mann–Whitney rank-biserial for Complexity is **negative** (hubs > non-hubs in mean Complexity, but negative rank-biserial because of how r_rb is defined — here, the sign indicates hubs tend to have *higher* values when r_rb is negative under your formula).
   - EPDC is flipped: MWU rank-biserial **positive** (~0.22), with **mean Complexity lower in hubs** (11.96 vs 12.84) and a **negative regression coefficient** (~−0.91). This is internally consistent and shows a genuinely inverse relationship not driven by Purity/UMI.
   - VSMC: mean Complexity of hubs >> non-hubs (10.77 vs 7.39), and both MWU (r_rb strongly negative) and regression (β ~ −0.74) show that, *after controlling for Purity/UMI*, hub status predicts **lower** Complexity relative to what would be expected given covariates. That is, raw and adjusted views tell different stories—this is one of the more interesting lineages mechanistically.

   This highlights that EPDC and VSMC are **non-trivial**: in VSMC, hubs look “highly complex” in absolute terms, but once Purity and UMI are accounted for, they are relatively downshifted in Complexity. This is exactly the kind of nuanced inverse entropy–Complexity pattern your hypothesis is pointing at.

3. **Lineage-wise pattern summary for later steps**

   For the downstream DE and interpretive steps, you now have a clear partition of lineages by **sign of entropy–Complexity coupling after adjustment**:

   - **Positive coupling lineages** (hubs higher Complexity given covariates):
     - vFibro, aFibro, adFibro
     - BEC, VEC
     - Pericyte
     - aEndocardial, vEndocardial
     - (VIC likely “weak/ambiguous” and could be set aside or treated as neutral)

   - **Negative coupling lineages** (hubs lower Complexity given covariates):
     - EPDC
     - VSMC

   This is exactly the grouping you need for step 3: “highlight whether lineages with positive versus negative entropy–Complexity coupling show distinct categories of DE genes.”  

4. **How to leverage these results for the next DE step**

   Concrete suggestions for step 3:

   a. **Define hub_status as already computed (per population) and reuse the same high-purity subset**  
   - Restrict to the same high-purity cells per population (≥30% quantile in Purity).
   - Use the existing `hub_status` label (0/1) you’ve generated per population.
   - Run `sc.tl.rank_genes_groups` within each focal population on this subset:  
     `sc.tl.rank_genes_groups(adata_pop, groupby='hub_status', method='wilcoxon', use_raw=False, pts=True)`  
     Make sure you know which group is “1” (hubs) in the Scanpy output.

   b. **Compare DE patterns between positive- and negative-coupling groups**

   For each population:
   - Extract top upregulated genes in hubs (e.g., top 20 by log2FC, with FDR < 0.05).
   - Summarize whether these genes are:
     - signaling/ligand/receptor/cell–cell interaction markers,
     - proliferation/stress/metabolic genes,
     - canonical lineage markers vs cross-lineage signatures.

   Then:
   - **Contrast positive vs negative coupling lineages**:
     - Do lineages with **positive** entropy–Complexity coupling (e.g., fibroblasts, endothelial, endocardial, pericytes) show hub markers enriched for **interaction/communication programs** (e.g., multiple ligands, receptors, ECM modifiers)?
     - Do **EPDC/VSMC hubs**, which show **negative** coupling, instead upregulate a **more restricted, possibly contractile or ECM-focused** program with *reduced* apparent transcriptomic breadth?

   Given the 140-gene panel, you won’t get full pathway coverage, but you can still test:
   - Are hubs enriched for genes from multiple functional “modules” (e.g., a mix of TFs + ligands + receptors + ECM) in positive lineages, versus a narrower, maybe more specialized module in EPDC/VSMC?
   - Are any canonical EPDC or VSMC markers *downregulated* in hubs if they indeed represent a more interaction-dominated, less internally complex state?

   c. **Relate DE patterns back to Complexity and covariates**

   For a small set of top hub-up genes per lineage:
   - Examine whether their expression correlates with Complexity and entropy within that lineage (simple Spearman, adjusted for Purity/UMI if desired).
   - Specifically in EPDC and VSMC:
     - Do hub-up genes show **negative association** with Complexity at fixed entropy/Purity/UMI?
     - This would help tie the inverse entropy–Complexity coupling to particular programs.

5. **Quality checks and potential refinements**

   - **VIC**: effect is weak; you might either:
     - Exclude VIC from “interface hub” biological interpretations, or
     - Treat it as a neutral/ambiguous lineage and see whether DE patterns look qualitatively different from both positive and negative groups.
   - **Purity confounding**:
     - You’ve adjusted for Purity in regression, which is good.
     - Still, for interpretability, you might summarize **Purity_mean_hub vs Purity_mean_nonhub** per lineage (already computed) to see whether hubs are consistently cleaner or more contaminated. This can help interpret whether complexity differences might be partially technical in appearance, even if statistically adjusted.

6. **Implications for the hypothesis**

   - The **first part of the hypothesis**—that entropy remains a lineage-specific predictor of Complexity after adjusting for Purity and UMI—is strongly supported by the regression-based hub_status effects, especially the coherent and consistent sign structure across lineages.
   - The **second part**—that high-entropy, reasonably pure “interface hubs” show distinct Complexity and covariate profiles, with **inverse patterns in EPDC and VSMC**—is very well supported:
     - EPDC and VSMC show statistically robust, **negative** hub_status coefficients despite being defined in the same way as other lineages.
     - Hubs in these lineages are distinct both in their raw and adjusted Complexity profiles, pointing to genuinely different interface programs.

   The upcoming DE analysis is now well-justified, and the most promising biological leverage will come from:
   - Comparing **hub vs non-hub DE** within each lineage.
   - Then **cross-comparing DE signatures** between positive vs negative entropy–Complexity lineages (especially EPDC/VSMC vs fibroblasts/endothelial/endocardial) to see whether the “interface hub” concept corresponds to distinct types of interface biology.

## Next Steps
Step 1: For each focal non-cardiomyocyte population, within the high-purity subset used for hub definition and using the pre-computed hub_status (hub vs non-hub), run within-population differential expression with sc.tl.rank_genes_groups (Wilcoxon, reference='rest') to obtain hub-upregulated markers, and summarize the top hub markers (names, log fold-changes, scores, and adjusted p-values) per population.
Step 2: Aggregate the per-population DE results into a text-only summary contrasting lineages with positive vs negative entropy–Complexity coupling by annotating each population with its coupling sign, tallying shared versus lineage-specific hub-upregulated genes within each coupling class, and qualitatively describing whether positive-coupling hubs display broader, multi-module transcriptional programs relative to the more restricted or distinct programs in inverse-coupling lineages (EPDC/VSMC).

## This code identifies “hub” cells within each non-cardiomyocyte population based on high spatial entropy, then performs within-population differential expression (Wilcoxon test) comparing hub vs non-hub cells to find genes upregulated in hubs. The goal is to systematically characterize transcriptional signatures of spatially complex/hub cells across lineages and summarize recurrent hub-associated genes.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

# We assume `adata` is already in memory and that spatial entropy and per-population
# hub assignments can be derived here if not already present. Here we:
# 1) Reuse the existing high-purity subset and hub definitions per focal population
#    if they exist.
# 2) If 'hub_status_global' is missing, we define a simple proxy hub label based on
#    spatial entropy within each focal population (top 20% entropy as hubs).
# 3) Run within-population DE (hub vs non-hub) using sc.tl.rank_genes_groups
#    with an explicit 'rest' reference and collect top hub-upregulated genes.

# Focal non-cardiomyocyte populations examined in the entropy/Complexity analysis
focal_pops = [
    'vFibro', 'aFibro', 'adFibro', 'EPDC', 'BEC', 'VEC',
    'VSMC', 'Pericyte', 'VIC', 'aEndocardial', 'vEndocardial'
]

# Columns that must be present from earlier steps
required_obs_cols = ['Populations', 'Purity', 'Complexity', 'UMI Count', 'spatial_entropy_k20']
for col in required_obs_cols:
    if col not in adata.obs.columns:
        raise ValueError(f"Required column '{col}' not found in adata.obs; cannot proceed with DE analysis.")

# Ensure numeric typing for relevant continuous covariates
for col in ['Purity', 'Complexity', 'UMI Count', 'spatial_entropy_k20']:
    adata.obs[col] = pd.to_numeric(adata.obs[col], errors='coerce')

# If hub_status_global is missing, define a simple entropy-based hub label within focal pops
if 'hub_status_global' not in adata.obs.columns:
    hub_status = pd.Series(np.nan, index=adata.obs.index, dtype=float)
    for pop in focal_pops:
        pop_mask = adata.obs['Populations'] == pop
        if pop_mask.sum() == 0:
            continue
        entropies = adata.obs.loc[pop_mask, 'spatial_entropy_k20']
        # require non-missing entropy
        entropies = entropies.dropna()
        if entropies.empty:
            continue
        # define hubs as top 20% of entropy values within this population
        thresh = np.quantile(entropies, 0.8)
        is_hub = (adata.obs.loc[pop_mask, 'spatial_entropy_k20'] >= thresh)
        hub_status.loc[is_hub.index] = is_hub.astype(int)
    adata.obs['hub_status_global'] = hub_status

# Container for per-population DE summaries
all_de_summaries = []

print("Running within-population DE (hub vs non-hub) for focal non-cardiomyocyte lineages using hub_status_global labels...\n")

for pop in focal_pops:
    pop_mask = (adata.obs['Populations'] == pop)
    n_pop = int(pop_mask.sum())
    if n_pop < 50:
        print(f"Skipping {pop}: only {n_pop} cells (require >= 50).")
        continue

    # Restrict to cells in this population that already have a non-missing hub_status_global label.
    valid_mask = pop_mask & adata.obs['hub_status_global'].notna()
    n_valid = int(valid_mask.sum())
    if n_valid < 30:
        print(f"{pop}: fewer than 30 cells with existing hub_status_global; skipping DE.")
        continue

    adata_sub = adata[valid_mask].copy()

    # Encode hub_status as categorical strings '0' and '1' (non-hub vs hub)
    adata_sub.obs['hub_status'] = adata_sub.obs['hub_status_global'].astype(int).astype(str)
    if adata_sub.obs['hub_status'].nunique() < 2:
        print(f"{pop}: only one hub_status group present in subset; skipping.")
        continue

    # Run DE: hubs ("1") vs non-hubs ("0"), Wilcoxon test, reference='rest' for clarity,
    # and consider all genes in the panel.
    sc.tl.rank_genes_groups(
        adata_sub,
        groupby='hub_status',
        method='wilcoxon',
        reference='rest',
        use_raw=False,
        n_genes=adata_sub.n_vars,
        pts=True
    )

    rgg = adata_sub.uns['rank_genes_groups']
    groups = list(rgg['names'].dtype.names)
    if '1' not in groups or '0' not in groups:
        print(f"{pop}: DE groups did not include both hub_status labels '0' and '1'; skipping summarization.")
        continue

    def _extract_group_df(rgg_dict, group_label, top_n=20):
        """Convert Scanpy rank_genes_groups results for a given group into a tidy DataFrame."""
        names = pd.Series(rgg_dict['names'][group_label], dtype=str)
        # logfoldchanges may not be present for Wilcoxon in some Scanpy versions.
        if 'logfoldchanges' in rgg_dict:
            lfc = pd.Series(rgg_dict['logfoldchanges'][group_label], dtype=float)
        else:
            lfc = pd.Series(np.nan, index=names.index)
        scores = pd.Series(rgg_dict['scores'][group_label], dtype=float)
        pvals = pd.Series(rgg_dict['pvals'][group_label], dtype=float)
        pvals_adj = pd.Series(rgg_dict['pvals_adj'][group_label], dtype=float)
        if 'pts' in rgg_dict and isinstance(rgg_dict['pts'], pd.DataFrame):
            pts = pd.Series(rgg_dict['pts'][group_label], dtype=float)
        else:
            pts = pd.Series(np.nan, index=names.index)

        df_out = pd.DataFrame({
            'gene': names.values,
            'logfoldchange': lfc.values,
            'score': scores.values,
            'pval': pvals.values,
            'pval_adj': pvals_adj.values,
            'pct_expressed_hub': pts.values
        })

        # Focus on genes up in hubs: prioritize positive logFC when available, else by score.
        if df_out['logfoldchange'].notna().any():
            df_out = df_out.sort_values(['logfoldchange', 'score'], ascending=[False, False])
        else:
            df_out = df_out.sort_values('score', ascending=False)
        return df_out.head(top_n).reset_index(drop=True)

    de_top_hubs = _extract_group_df(rgg, group_label='1', top_n=20)
    de_top_hubs['Population'] = pop
    de_top_hubs['hub_group'] = '1'  # hubs
    all_de_summaries.append(de_top_hubs)

    print(f"{pop}: DE completed on N={n_valid} high-purity cells with existing hub labels.")
    print("Top hub-upregulated genes (first 10):")
    print(de_top_hubs.head(10).to_string(index=False, float_format=lambda x: f"{x:.3g}"))
    print("\n")

# After loop: aggregate and print a concise cross-lineage summary of hub-upregulated genes
if all_de_summaries:
    de_all_df = pd.concat(all_de_summaries, axis=0, ignore_index=True)

    print("\n=== Aggregated summary of hub-upregulated genes across focal populations ===")
    # For each gene, count in how many populations it appears among the top hub-up markers
    gene_counts = (
        de_all_df.groupby('gene')['Population']
        .nunique()
        .sort_values(ascending=False)
    )
    print("\nGenes most frequently upregulated in hubs across focal lineages (top 20):")
    print(gene_counts.head(20).to_string())

    # Per-population, show the top 5 hub-up genes by logFC (or score if logFC missing)
    print("\nPer-population top 5 hub-upregulated genes:")
    for pop in sorted(de_all_df['Population'].unique()):
        df_pop = de_all_df[de_all_df['Population'] == pop].copy()
        if df_pop.empty:
            continue
        df_top5 = df_pop.head(5)
        print(f"\nPopulation: {pop}")
        print(df_top5[['gene', 'logfoldchange', 'score', 'pval_adj', 'pct_expressed_hub']]
              .to_string(index=False, float_format=lambda x: f"{x:.3g}"))
else:
    print("No DE results were generated; check population sizes and hub labels.")

Running within-population DE (hub vs non-hub) for focal non-cardiomyocyte lineages using hub_status_global labels...

ranking genes


    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:01)


vFibro: DE completed on N=16624 high-purity cells with existing hub labels.
Top hub-upregulated genes (first 10):
  gene  logfoldchange  score     pval  pval_adj  pct_expressed_hub Population hub_group
 MYH11          0.458  0.979    0.327     0.595              0.389     vFibro         1
ELAVL2          0.345   0.82    0.412     0.675              0.338     vFibro         1
  ASPN          0.335   2.67  0.00754    0.0332              0.118     vFibro         1
 TENM2          0.323   2.38   0.0173    0.0707              0.051     vFibro         1
PIEZO2          0.274   1.87   0.0619     0.173              0.819     vFibro         1
 F13A1          0.256   1.11    0.269     0.529              0.491     vFibro         1
 LYVE1          0.253   1.24    0.215     0.449              0.926     vFibro         1
  OSR1          0.248   2.15   0.0315     0.106              0.057     vFibro         1
IGFBP4          0.246   8.42 3.82e-17  4.54e-15               0.37     vFibro         1
 TENM3

    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:00)


aFibro: DE completed on N=7417 high-purity cells with existing hub labels.
Top hub-upregulated genes (first 10):
  gene  logfoldchange  score     pval  pval_adj  pct_expressed_hub Population hub_group
  DLK1           1.56   16.4 3.06e-60  2.43e-58              0.283     aFibro         1
  OSR1           1.11   12.9 5.01e-38  2.98e-36              0.159     aFibro         1
  NEFL           1.05    1.6    0.109     0.178              0.428     aFibro         1
  ISL1          0.868  0.978    0.328     0.429              0.146     aFibro         1
ENTPD2          0.712   2.91  0.00362   0.00797                0.1     aFibro         1
  PRPH          0.661  0.275    0.783     0.836              0.651     aFibro         1
 CLDN5          0.657  0.842      0.4     0.501              0.293     aFibro         1
  IRX4          0.643  0.796    0.426     0.528              0.773     aFibro         1
 LYVE1          0.534  0.488    0.625     0.729             0.0314     aFibro         1
 PTCH2 

...

VIC: DE completed on N=11596 high-purity cells with existing hub labels.
Top hub-upregulated genes (first 10):
   gene  logfoldchange  score      pval  pval_adj  pct_expressed_hub Population hub_group
   MYH7           1.37   35.6 2.62e-277 6.23e-275              0.176        VIC         1
   DLK1           1.09   5.56  2.77e-08  1.33e-07              0.145        VIC         1
COLEC11           1.06   8.01  1.11e-15  7.99e-15              0.217        VIC         1
   OSR1           1.05    7.7  1.39e-14  9.44e-14              0.287        VIC         1
    DES          0.939   22.6 7.29e-113 5.78e-111              0.169        VIC         1
  SCN7A          0.907   8.71  3.13e-18  2.86e-17               0.12        VIC         1
 CXCL12          0.876   11.5  1.15e-30  1.82e-29              0.173        VIC         1
   IRX4          0.866   5.66  1.53e-08  7.58e-08              0.503        VIC         1
   MYH6          0.821   17.3   6.1e-67  2.07e-65              0.574        VIC

    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:00)


aEndocardial: DE completed on N=4599 high-purity cells with existing hub labels.
Top hub-upregulated genes (first 10):
   gene  logfoldchange  score     pval  pval_adj  pct_expressed_hub   Population hub_group
  RSPO3           1.14   2.27    0.023    0.0548               0.15 aEndocardial         1
SLC26A7          0.948   4.49 7.11e-06  3.84e-05              0.808 aEndocardial         1
    NDP          0.914  0.796    0.426     0.596              0.164 aEndocardial         1
   PENK          0.855   2.57   0.0101    0.0265             0.0723 aEndocardial         1
    NTS          0.854    3.4 0.000681   0.00232             0.0387 aEndocardial         1
  SULF1          0.773      6 1.95e-09  2.11e-08              0.526 aEndocardial         1
  FBLN2          0.768   10.8 5.33e-27  3.17e-25              0.116 aEndocardial         1
    CPE          0.666   3.06  0.00224   0.00692              0.592 aEndocardial         1
   DLK1            0.6   1.63    0.104     0.198              

    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:00)


vEndocardial: DE completed on N=10887 high-purity cells with existing hub labels.
Top hub-upregulated genes (first 10):
 gene  logfoldchange  score     pval  pval_adj  pct_expressed_hub   Population hub_group
DHRS3          0.543   9.54 1.38e-21   4.7e-20              0.293 vEndocardial         1
 GAS7          0.386   5.75 9.06e-09  8.29e-08               0.48 vEndocardial         1
 MYH6          0.372   9.49 2.32e-21   6.9e-20              0.628 vEndocardial         1
CKMT2          0.338   7.61 2.84e-14  6.75e-13              0.427 vEndocardial         1
 MYH7          0.313   12.1 1.39e-33   1.1e-31               0.32 vEndocardial         1
ABCC9          0.306   3.54 0.000403   0.00188              0.512 vEndocardial         1
  DES          0.302     10 1.51e-23  7.16e-22              0.194 vEndocardial         1
MYBL2          0.291   4.94  7.7e-07  4.95e-06              0.918 vEndocardial         1
 RGS5          0.281   4.13 3.65e-05  0.000204              0.389 vEndocardial 

### Agent Interpretation

These DE results give you a strong starting point for testing the hub–non‑hub transcriptional differences, but they don’t yet speak directly to the “positive vs negative entropy–Complexity coupling” contrast in the hypothesis. A few key points and suggestions:

1. **Hub vs non‑hub contrast is clearly non‑trivial.**  
   - Within essentially every lineage you tested, hubs have robustly upregulated genes (often with large logFC and extremely small adjusted p‑values), so the basic premise that “interface hubs are transcriptionally distinct” is supported.
   - Many hub markers are plausible “interface” or transition signatures: ECM/collagen (FNDC1, COL26A1, COL15A1), Notch/Hey (JAG1, HEY1), endothelial/vasculogenic markers in VSMC hubs (CD34, CLDN5, PECAM1, NOS3), epicardial/EPDC signatures in fibroblast-like hubs (TCF21, ALDH1A2, ITLN1, FBLN5).

2. **A very strong shared program centered on sarcomeric / contractile and metabolic genes.**  
   - MYH7 is top or near‑top in 9/11 populations; CKMT2, DES, CASQ2, and MYH6 also show up repeatedly. This suggests a generic “cardiomyocyte‑like/contractile” or “stress/contractility/metabolic” program in hubs across many lineages (fibroblast, endothelial, VIC, EPDC, VSMC, endocardial).
   - This makes hubs look like “interface” states between non‑CM lineages and cardiomyocyte identity or mechanical coupling, which is biologically interesting and consistent with high entropy.

3. **Lineage‑specific flavors overlaying this shared program.**  
   Some examples from the top‑5 summaries:
   - **Fibroblasts (vFibro, aFibro, adFibro):**  
     - aFibro/adFibro hubs: DLK1, OSR1, TCF21, ALDH1A2, COL15A1, TECRL, PRPH. Strong epicardial/valve/fibroblast‑progenitor / developmental TF signature, plus ECM.  
     - vFibro hubs: ASPN, OSR1, IGFBP4, PIEZO2, LYVE1 – more ECM/mechano‑sensing and perivascular/lymphatic markers.
   - **EPDC (inverse‑coupling candidate):**  
     - Very strong, coherent hub program: FNDC1, NKD2, SCN7A, COL26A1, FBLN5, ITLN1, DHRS3, INMT, F13A1, CPE. This is ECM + Wnt/retinoic acid/secreted factors + neuropeptide processing — a multipronged program.
   - **VSMC (inverse‑coupling candidate):**  
     - Hubs: CD34, CLDN5, PECAM1, NOS3, HEY1, GJA5, ABCC9, CAV1, PGF – striking endothelial / arterial / shear‑responsive program in “smooth muscle” hubs; highly multi‑module (endothelial identity, Notch/Hey, angiogenic factors, ion channels).
   - **Endothelial / Endocardial (positive‑coupling candidates):**  
     - BEC hubs: MYH7, FRZB, COL15A1, LBH, PCNA – mild contractile + Wnt modulator + proliferation.  
     - VEC hubs: NPR3, MYH7, FN1, DES, CKMT2, CASQ2, NR2F2 – contractile/metabolic with natriuretic peptide receptor and ECM.  
     - a/vEndocardial hubs: RSPO3, SLC26A7, SULF1, FBLN2, DHRS3, GAS7, CKMT2, MYH6/7 – Wnt modulators, ECM, metabolic/contractile, and ion handling.

   So, qualitatively, hubs in all lineages are not just “randomly different”; they carry meaningful programs—often multi‑module—related to ECM, vascular/endothelial identity, developmental signaling, and cardiac contractility.

4. **Where this stands relative to the *specific* hypothesis (positive vs negative coupling).**  
   To connect these DE results to entropy–Complexity coupling:
   - You need an annotation per population: **coupling sign (positive vs negative)** from your prior regression/partial correlation analysis.
   - The hypothesis predicts that:
     - **Positive‑coupling lineages** (e.g. fibroblast/endothelial/endocardial) will have hub programs that are perhaps broad but in one “direction” (e.g., joint increase of entropy and Complexity with a shared contractile/ECM program).  
     - **Negative‑coupling lineages** (EPDC, VSMC) will have **systematically different** hub programs — maybe more “inverse” features (e.g., gaining interface identity/vascular signatures while losing Complexity, or having more discrete, specialized modules).

   From the current text alone:
   - EPDC/VSMC hubs indeed look **very multi‑module and “switch‑like”**:  
     - EPDC hubs strongly upregulate a tight set of ECM + Wnt/RA + secreted factor genes with huge effect sizes.  
     - VSMC hubs show a strong endothelialization/arborization program (CD34, CLDN5, PECAM1, NOS3, PGF, HEY1, GJA5) with enormous logFC.  
   - Positive‑coupling fibroblast/endothelial/endocardial hubs show:
     - A more **shared cardiomyocyte‑like contractile/metabolic overlay (MYH7, CKMT2, DES)** plus lineage‑specific developmental factors (DLK1/OSR1/TCF21 in fibroblasts; RSPO3/SULF1/FBLN2 in aEndocardial; FRZB/NPR3/LBH in endothelia).

   Qualitatively, that already hints at the pattern you want: inverse‑coupling hubs (EPDC/VSMC) are not just “more of the same positive‑coupling program” but rather shift toward endothelial/ECM/secreted developmental states that are distinct from the more uniform contractile overlay seen in the positive‑coupling lineages.

5. **Concrete next steps to make this hypothesis testable and distinct from the paper / past analyses:**

   a. **Explicitly label each population with coupling sign and summarize hub programs by sign.**  
   - Bring in your previous entropy–Complexity regression results and create a small mapping:  
     `coupling_sign = {'EPDC': 'negative', 'VSMC': 'negative', 'vFibro': 'positive', ...}`  
   - Merge this with `de_all_df`.  
   - For each **sign class**, compute:
     - The list of hub‑upregulated genes and their counts across populations (like your `gene_counts`, but stratified by sign).  
     - Which genes are **shared only within positive lineages**, only within negative, or across both.

   b. **Quantify breadth vs specificity of the hub program per sign.**  
   - For each population, compute:
     - The number of significantly upregulated genes in hubs (FDR < 0.05, logFC > some threshold).  
     - The Gini index / entropy of the effect size distribution (are hubs driven by a few huge hits or many moderate ones?).  
   - Compare these summary statistics between positive‑coupling and negative‑coupling lineages.  
   - This addresses the “broader, multi‑module vs restricted” claim quantitatively.

   c. **Module‑level characterization without going beyond the dataset.**  
   Using only the 140‑gene panel:
   - Define a few coarse “modules” (manually, based on known roles of these genes, not external datasets), e.g.:  
     - Contractile / sarcomeric / ion‑handling (MYH6, MYH7, DES, CASQ2, PLN, CKMT2, etc.)  
     - ECM / fibroblast / EPDC (FN1, COL15A1, COL26A1, DCN, FBLN2/5, FNDC1, ASPN, etc.)  
     - Vascular/endothelial (CLDN5, PECAM1, CD34, NOS3, PGF, NR2F2, GJA5, NPR3, LYVE1, etc.)  
     - Developmental signaling / TFs (DLK1, OSR1, TCF21, NKD2, RSPO3, FRZB, HEY1, NDP, SULF1, LBH, etc.)  
   - For each population and for hubs vs non‑hubs, compute module scores (mean z‑score of genes in that module).  
   - Then, for **positive vs negative coupling classes**, compare which modules are preferentially up in hubs:
     - Do positive‑coupling fibroblast/endothelial/endocardial hubs mostly increase **contractile + ECM** scores?  
     - Do negative‑coupling EPDC/VSMC hubs show a more drastic shift to **endothelial/vascular + developmental** modules?

   d. **Check purity/UMI confounding post‑hoc.**  
   - You adjusted for Purity and UMI Count in the earlier coupling analysis, but here the DE is unconditional.  
   - At minimum: compare Purity and UMI distributions between hubs and non‑hubs within each population to confirm no massive confounding. If there is, you might want to re‑run DE using a method that includes covariates (e.g., pseudo‑bulk per sample * hub_status; or a generalized linear model via diffxpy/MAST‑like approach).

   e. **Visual integration with spatial/entropy context (to connect back to hub definition).**  
   - For key hub markers that differ by coupling sign (e.g., MYH7, CKMT2 vs CD34/CLDN5/FNDC1/NKD2/HEY1), plot:
     - UMAP colored by expression plus overlay of hub vs non‑hub.  
     - Tissue plots showing where those hub cells sit spatially, ideally annotated by lineage and hub_status.  
   - This helps interpret whether negative‑coupling hubs occupy distinct physical niches where endothelialization or EPDC activation happens.

6. **Genes that appear particularly informative for the sign‑specific contrast (good candidates for follow‑up):**

   - **Shared “global hub” markers across lineages:** MYH7, CKMT2, DES, CASQ2, SLC26A7, F13A1, CLDN5, ABCC9, OSR1, DLK1, FBLN5, SCN7A. These will likely define a “core hub” signature.
   - **Enriched in negative‑coupling (EPDC/VSMC) hubs:**  
     - EPDC: FNDC1, NKD2, COL26A1, ITLN1, DHRS3, INMT, FBLN5, CPE.  
     - VSMC: CD34, CLDN5, GJA5, HEY1, PECAM1, NOS3, PGF, CAV1.  
   - **Enriched in positive‑coupling fibroblast / endothelial / endocardial hubs:**  
     - Fibroblast: TCF21, ALDH1A2, COL15A1, ASPN, IGFBP4, PRRX1, TECRL.  
     - Endothelial/Endocardial: RSPO3, FRZB, SULF1, FBLN2, LBH, NPR3, NR2F2, LYVE1.

   These sets can be contrasted directly (heatmaps, module scores) between coupling classes to see if negative‑coupling hubs emphasize vascularization/secreted ECM‑signaling, while positive‑coupling hubs emphasize a graded cardiomyocyte‑like/contractile spectrum.

In summary, the DE step successfully shows that entropy‑defined hubs have distinct, biologically interpretable transcriptional programs, with a strong shared contractile/ECM core and clear lineage‑specific overlays. To really test and formalize the hypothesis about **systematic differences between positive‑ vs negative‑coupling lineages**, your next steps should (i) annotate each population by coupling sign, (ii) stratify and quantify shared vs sign‑specific hub markers and modules, and (iii) summarize at the module level how hub programs differ between EPDC/VSMC and fibroblast/endothelial/endocardial lineages.

## Next Steps
Step 1: Aggregate the per-population hub vs non-hub differential expression results into a single table annotated with each lineage’s entropy–Complexity coupling sign (positive, negative, or neutral), ensuring that hub labels exactly match those used in the prior coupling and hub-status analyses (top 10% spatial entropy within top 70% Purity per population), and generate a text-only summary that (i) tallies genes recurrently upregulated in hubs within positive-coupling lineages versus negative-coupling lineages, (ii) identifies genes uniquely hub-upregulated in negative-coupling lineages (EPDC/VSMC) versus fibroblast/endothelial/endocardial lineages, and (iii) qualitatively classifies these hub programs into coarse functional modules (contractile/ion-handling, ECM/fibroblast, endothelial/vascular, developmental/signaling) to assess whether positive-coupling hubs display broader, multi-module activation compared with more specialized or distinct programs in inverse-coupling lineages.

## This code rebuilds the hub vs non-hub differential expression table using a hub definition that exactly matches the prior entropy–Complexity coupling analysis (top 10% spatial entropy within the top 70% Purity per population), annotates each lineage with its entropy–Complexity coupling sign from earlier regressions, and produces text-only summaries of recurrent hub-upregulated genes and their distribution across positive, negative, and neutral coupling classes, including a coarse functional module classification.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

# This step assumes that:
#  - `adata` is in memory and `.X` contains log-normalized expression used in prior DE.
#  - `spatial_entropy_k20` and `hub_status_global_DE` were defined earlier using the
#    *same* hub definition as used for entropy–Complexity coupling:
#       top 10% of spatial entropy (spatial_entropy_k20) within the top 70% Purity
#       of each focal population (per-population 0.3 Purity quantile as floor).
#  - The coupling sign per lineage was derived from the prior regression/partial
#    correlation analysis of Complexity ~ entropy + Purity + UMI Count.

# Define coupling sign per lineage based on the earlier regression results:
#   positive  = hubs higher Complexity at fixed Purity & UMI
#   negative  = hubs lower  Complexity at fixed Purity & UMI
#   neutral   = weak/ambiguous or non-significant coupling
coupling_sign_map = {
    'vFibro': 'positive',
    'aFibro': 'positive',
    'adFibro': 'positive',
    'BEC': 'positive',
    'VEC': 'positive',
    'Pericyte': 'positive',
    'aEndocardial': 'positive',
    'vEndocardial': 'positive',
    'VIC': 'neutral',
    'EPDC': 'negative',
    'VSMC': 'negative',
}

focal_pops = list(coupling_sign_map.keys())

# Sanity checks
required_obs_cols = ['Populations', 'Purity', 'spatial_entropy_k20']
for col in required_obs_cols:
    if col not in adata.obs.columns:
        raise ValueError(f"Required column '{col}' not found in adata.obs; cannot proceed.")

# Ensure numeric typing for Purity and entropy
for col in ['Purity', 'spatial_entropy_k20']:
    adata.obs[col] = pd.to_numeric(adata.obs[col], errors='coerce')

# (Re)define per-population hub_status_global_DE *exactly matching* the earlier
# hub definition used for entropy–Complexity coupling: per-population top 10%
# spatial entropy within the top 70% Purity.
hub_status_global = pd.Series(np.nan, index=adata.obs.index, dtype=float)

for pop in focal_pops:
    pop_mask = adata.obs['Populations'] == pop
    n_pop = int(pop_mask.sum())
    if n_pop < 50:
        continue

    # Work within this population, drop NA in key metrics
    df_pop = adata.obs.loc[pop_mask, ['Purity', 'spatial_entropy_k20']].copy()
    df_pop = df_pop.dropna(subset=['Purity', 'spatial_entropy_k20'])
    if df_pop.shape[0] < 50:
        continue

    # Per-population purity floor (bottom 30% removed, i.e. retain top 70% Purity)
    purity_floor = df_pop['Purity'].quantile(0.3)
    high_purity_idx = df_pop.index[df_pop['Purity'] >= purity_floor]
    if len(high_purity_idx) < 40:
        continue

    # Within the retained high-purity cells, define hubs as top 10% of spatial entropy
    ent_vals = df_pop.loc[high_purity_idx, 'spatial_entropy_k20']
    if ent_vals.isna().all():
        continue
    entropy_thresh = ent_vals.quantile(0.9)
    hubs = ent_vals >= entropy_thresh

    hub_status_global.loc[high_purity_idx] = hubs.astype(int).values

adata.obs['hub_status_global_DE'] = hub_status_global

# Helper to extract a tidy hub-upregulated DE table from Scanpy rank_genes_groups

def extract_group_df(rgg_dict, group_label, top_n=200):
    names = pd.Series(rgg_dict['names'][group_label], dtype=str)
    if 'logfoldchanges' in rgg_dict:
        lfc = pd.Series(rgg_dict['logfoldchanges'][group_label], dtype=float)
    else:
        lfc = pd.Series(np.nan, index=names.index)
    scores = pd.Series(rgg_dict['scores'][group_label], dtype=float)
    pvals_adj = pd.Series(rgg_dict['pvals_adj'][group_label], dtype=float)
    # pts (fraction expressing in group) may or may not be present
    if 'pts' in rgg_dict and isinstance(rgg_dict['pts'], pd.DataFrame):
        pts = pd.Series(rgg_dict['pts'][group_label], dtype=float)
    else:
        pts = pd.Series(np.nan, index=names.index)

    df_out = pd.DataFrame({
        'gene': names.values,
        'logfoldchange': lfc.values,
        'score': scores.values,
        'pval_adj': pvals_adj.values,
        'pct_expressed_hub': pts.values,
    })

    # Keep genes that are genuinely hub-upregulated (positive logFC where available)
    if df_out['logfoldchange'].notna().any():
        df_out = df_out[df_out['logfoldchange'] > 0]
        df_out = df_out.sort_values(['logfoldchange', 'score'], ascending=[False, False])
    else:
        df_out = df_out.sort_values('score', ascending=False)

    return df_out.head(top_n).reset_index(drop=True)

# Run DE per population using the matched hub_status_global_DE and build a combined table
all_de_rows = []

for pop in focal_pops:
    pop_mask = (adata.obs['Populations'] == pop)
    n_pop = int(pop_mask.sum())
    if n_pop < 50:
        continue

    valid_mask = pop_mask & adata.obs['hub_status_global_DE'].notna()
    n_valid = int(valid_mask.sum())
    if n_valid < 30:
        continue

    adata_sub = adata[valid_mask].copy()

    # Hub status as categorical string for Scanpy
    adata_sub.obs['hub_status'] = adata_sub.obs['hub_status_global_DE'].astype(int).astype(str)
    if adata_sub.obs['hub_status'].nunique() < 2:
        continue

    sc.tl.rank_genes_groups(
        adata_sub,
        groupby='hub_status',
        method='wilcoxon',
        reference='rest',
        use_raw=False,
        n_genes=adata_sub.n_vars,
        pts=True,
    )

    rgg = adata_sub.uns['rank_genes_groups']
    groups = list(rgg['names'].dtype.names)
    if '1' not in groups:
        continue

    de_top = extract_group_df(rgg, group_label='1', top_n=200)
    de_top['Population'] = pop
    de_top['coupling_sign'] = coupling_sign_map.get(pop, 'unknown')
    all_de_rows.append(de_top)

if not all_de_rows:
    print("No DE results generated for focal populations; check hub assignments.")
else:
    de_all = pd.concat(all_de_rows, axis=0, ignore_index=True)

    # Filter for statistically significant hub-up genes (FDR < 0.05)
    sig = de_all[de_all['pval_adj'] < 0.05].copy()

    # Split by coupling sign
    pos_sig = sig[sig['coupling_sign'] == 'positive']
    neg_sig = sig[sig['coupling_sign'] == 'negative']
    neu_sig = sig[sig['coupling_sign'] == 'neutral']

    print("\n=== Hub-upregulated genes stratified by entropy–Complexity coupling sign ===\n")

    # 1) Genes recurrently hub-up in positive-coupling lineages
    if not pos_sig.empty:
        pos_counts = (
            pos_sig.groupby('gene')['Population']
            .nunique()
            .sort_values(ascending=False)
        )
        print("Genes most frequently hub-upregulated in positive-coupling lineages (top 20):")
        print(pos_counts.head(20).to_string())
    else:
        print("No significant hub-up genes detected in positive-coupling lineages.")

    # 2) Genes recurrently hub-up in negative-coupling lineages (EPDC/VSMC)
    if not neg_sig.empty:
        neg_counts = (
            neg_sig.groupby('gene')['Population']
            .nunique()
            .sort_values(ascending=False)
        )
        print("\nGenes most frequently hub-upregulated in negative-coupling lineages (EPDC/VSMC, top 20):")
        print(neg_counts.head(20).to_string())
    else:
        print("\nNo significant hub-up genes detected in negative-coupling lineages.")

    # 3) Identify genes unique to negative-coupling hubs vs those shared with positive-coupling hubs
    pos_genes = set(pos_sig['gene'].unique())
    neg_genes = set(neg_sig['gene'].unique())
    neg_unique = sorted(neg_genes - pos_genes)
    pos_unique = sorted(pos_genes - neg_genes)
    shared = sorted(pos_genes & neg_genes)

    print("\nCounts of hub-up genes by overlap class:")
    print(f"  Positive-only hub-up genes: {len(pos_unique)}")
    print(f"  Negative-only hub-up genes: {len(neg_unique)}")
    print(f"  Shared between positive and negative lineages: {len(shared)}")

    if neg_unique:
        print("\nExample negative-only hub-up genes (up to 20):")
        print("  " + ", ".join(neg_unique[:20]))
    if pos_unique:
        print("\nExample positive-only hub-up genes (up to 20):")
        print("  " + ", ".join(pos_unique[:20]))
    if shared:
        print("\nExample shared hub-up genes (up to 20):")
        print("  " + ", ".join(shared[:20]))

    # 4) Very coarse module classification of hub-up genes by hard-coded membership
    contractile_genes = {
        'MYH6', 'MYH7', 'DES', 'CASQ2', 'CKMT2', 'PLN', 'TNNT2', 'TTN', 'RYR2',
    }
    ecm_fibro_genes = {
        'FN1', 'FNDC1', 'COL15A1', 'COL26A1', 'DCN', 'FBLN2', 'FBLN5', 'ASPN', 'POSTN',
    }
    endothelial_genes = {
        'CLDN5', 'PECAM1', 'CD34', 'NOS3', 'PGF', 'NR2F2', 'GJA5', 'NPR3', 'CAV1', 'LYVE1',
    }
    dev_signaling_genes = {
        'DLK1', 'OSR1', 'TCF21', 'NKD2', 'RSPO3', 'FRZB', 'HEY1', 'SULF1', 'LBH', 'NDP',
    }

    def summarize_modules(sub_df, label):
        genes = set(sub_df['gene'].unique())
        print(f"\nModule-level presence in {label} hub-up genes:")
        print(f"  Contractile/ion-handling genes: {sorted(contractile_genes & genes)}")
        print(f"  ECM/fibroblast genes:          {sorted(ecm_fibro_genes & genes)}")
        print(f"  Endothelial/vascular genes:    {sorted(endothelial_genes & genes)}")
        print(f"  Developmental/signaling genes: {sorted(dev_signaling_genes & genes)}")

    if not pos_sig.empty:
        summarize_modules(pos_sig, 'positive-coupling lineages')
    if not neg_sig.empty:
        summarize_modules(neg_sig, 'negative-coupling lineages (EPDC/VSMC)')
    if not neu_sig.empty:
        summarize_modules(neu_sig, 'neutral-coupling lineages (e.g., VIC)')


ranking genes


    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:00)


ranking genes


    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:00)


ranking genes


...

ranking genes


    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:00)


ranking genes


    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:00)



=== Hub-upregulated genes stratified by entropy–Complexity coupling sign ===

Genes most frequently hub-upregulated in positive-coupling lineages (top 20):
gene
FN1       4
CXCL12    3
DCN       3
FBLN5     3
NR2F2     3
MYH7      3
MYH6      3
TCF21     3
FNDC1     2
DLK1      2
LBH       2
IGFBP5    2
CPE       2
DES       2
CD34      2
MCAM      2
PRSS35    2
OSR1      2
NOTCH1    2
SOX9      2

Genes most frequently hub-upregulated in negative-coupling lineages (EPDC/VSMC, top 20):
gene
ARHGAP18    2
ALDH1A2     2
ADM         2
CDT1        2
CAV1        2
CASQ2       2
ARHGAP29    2
ARL6IP1     2
BTG1        2
FOXS1       2
FGF12       2
CPE         2
CPNE3       2
CTSC        2
DCN         2
DES         2
COLEC11     2
COL15A1     2
CKMT2       2
RGS5        2

Counts of hub-up genes by overlap class:
  Positive-only hub-up genes: 20
  Negative-only hub-up genes: 104
  Shared between positive and negative lineages: 43

Example negative-only hub-up genes (up to 20):
  ABCC9, ADAMT

### Agent Interpretation

These results are very informative and generally supportive of the hypothesis, but they also highlight some subtleties you’ll want to unpack in the next steps.

Key takeaways for the hypothesis
--------------------------------

1. **There is a clear, lineage-stratified hub signature.**
   - You successfully enforced the same hub definition used for entropy–Complexity coupling, so the DE is conceptually aligned with the coupling analysis.
   - Both positive- and negative-coupling lineages yield sizeable sets of hub-upregulated genes with FDR < 0.05, meaning the “interface hub” concept is transcriptionally non-trivial in multiple lineages.

2. **Positive-coupling hubs show a recognizable “interface / multipotent fibro-vascular” program.**
   - Recurrently hub-up in positive-coupling lineages: `FN1`, `CXCL12`, `DCN`, `FBLN5`, `NR2F2`, `MYH6/7`, `TCF21`, `FNDC1`, `DLK1`, `LBH`, `CD34`, `MCAM`, `OSR1`, `NOTCH1`, `SOX9`, etc.
   - Your coarse module summary for positive-coupling hubs shows **representation across all four modules**:
     - Contractile/ion: `MYH6`, `MYH7`, `DES`
     - ECM/fibroblast: `ASPN`, `COL15A1`, `DCN`, `FBLN2`, `FBLN5`, `FN1`, `FNDC1`, `POSTN`
     - Endothelial/vascular: `CAV1`, `CD34`, `LYVE1`, `NPR3`, `NR2F2`
     - Developmental/signaling: `DLK1`, `LBH`, `NKD2`, `OSR1`, `SULF1`, `TCF21`
   - Conceptually, that matches “high-entropy, high-Complexity hubs” as **multi-module, interface-like cells** drawing from fibroblast, vascular, and developmental programs, rather than a single specialized axis.

3. **Negative-coupling hubs (EPDC/VSMC) are not “simpler”; they show strong, partly distinct programs.**
   - Recurrent negative-coupling hub genes include **both**:
     - ECM/vascular/EPDC-type genes: `ARHGAP18`, `ARHGAP29`, `ALDH1A2`, `CAV1`, `RGS5`, `COL15A1`, `CPE`, `CPNE3`, `COLEC11`
     - Clear contractile / ion-handling genes: `CASQ2`, `CKMT2`, `PLN`, plus `RYR2`, `TTN` from the module summary.
   - Module-level results for negative-coupling hubs show **broad coverage as well**:
     - Contractile/ion: `CASQ2`, `CKMT2`, `DES`, `MYH7`, `PLN`, `RYR2`, `TTN`
     - ECM/fibro: `ASPN`, `COL15A1`, `COL26A1`, `DCN`, `FBLN2`, `FBLN5`, `FNDC1`
     - Endothelial/vascular: `CAV1`, `CD34`, `CLDN5`, `GJA5`, `LYVE1`, `NOS3`, `NR2F2`, `PECAM1`, `PGF`
     - Developmental/signaling: `DLK1`, `FRZB`, `HEY1`, `LBH`, `NKD2`, `SULF1`, `TCF21`
   - So EPDC/VSMC hubs are not just “sharply specialized”; they too are **multi-module**, albeit with a different emphasis: more **VSMC-like contractile/ion-handling** (e.g. `CASQ2`, `PLN`, `RYR2`, `TTN`, `CKMT2`) and retinoic/TGF/Wnt-adjacent components (`ALDH1A2`, `ARHGAP18/29`, `FRZB`, `HEY1`).

4. **Overlap structure: negative hubs have many unique genes.**
   - Positive-only hub-up genes: 20
   - Negative-only hub-up genes: 104
   - Shared: 43
   - Examples:
     - **Positive-only**: `CXCL12`, `FN1`, `APOE`, `POSTN`, `NPR3`, `SOX9`, `TBX18`, `MYH6`, `VCAN`, `INHBA`
     - **Negative-only**: `ADM`, `ALDH1A2`, `ARHGAP18/29`, `ABCC9`, `BTG1`, `CD9`, `CLDN5`, `COL26A1`, `CASQ2`, `CKMT2`, `RGS5`
   - Many negative-only genes are quite **VSMC/EPDC/vascular/contractile signaling**; many positive-only genes are more **stromal / chemokine / interface** (`CXCL12`, `POSTN`, `NPR3`, `VCAN`).

   This supports the “systematically different” part of the hypothesis: the **content** of hub programs clearly differs between positive vs negative coupling lineages, even if both are multi-module in a coarse sense.

How this bears on the precise hypothesis
----------------------------------------

- The hypothesis predicts:
  1. Hub vs non-hub transcriptional differences within each lineage.
  2. Systematic differences in hub programs between lineages with **positive** vs **negative** entropy–Complexity coupling.
  3. Positive-coupling hubs having **broader, multi-module activation** versus more specialized programs in inverse-coupling lineages.

- From this step:
  - (1) is clearly met: both coupling classes have robust hub-up signatures.
  - (2) is partially validated: the **gene identities and emphasis** differ (e.g. `CXCL12/FN1/POSTN/NPR3` vs `ALDH1A2/ARHGAP18/29/ADM/CASQ2/CKMT2/RGS5`), with a substantial pool of negative-only genes (104).
  - (3) is **not clearly supported by the current coarse module counting**:
    - Both positive and negative hubs show presence across all four modules with this limited panel.
    - Negative hubs actually look **as or more multi-module** by this binary presence/absence summary.

  The most likely explanation is that with a 140-gene panel and hard-coded marker sets, “module breadth” by simple membership is too coarse: subtle quantitative balance within modules (e.g. contractile-heavy vs ECM-heavy vs endothelial-heavy) matters more than whether at least one gene from each category is present.

Promising directions for next steps
-----------------------------------

To move beyond these counts and more rigorously test the “broad multi-module vs specialized” aspect, I’d suggest:

1. **Quantify module *weights*, not just presence/absence, by coupling sign.**
   - For each module (contractile, ECM/fibro, endothelial, dev/signaling), compute:
     - Per-gene average logFC (hub vs non-hub) within each population.
     - Then per-module, per-population **mean or median logFC**.
   - Compare distributions of these module-level logFCs between:
     - Positive vs negative lineages (e.g., boxplots or violin plots).
   - Hypothesis-friendly signatures:
     - Positive-coupling hubs: more **balanced** activation across modules (module logFCs closer together).
     - Negative-coupling hubs: **skewed** activation (e.g., contractile/ion high, ECM/dev moderate, endothelial minimal or vice versa).

2. **Project modules at the single-cell level, stratified by hub status and coupling sign.**
   - Compute simple module scores per cell, using genes from your curated sets (or a slightly expanded set chosen from the DE results):
     - Example: `sc.tl.score_genes` for each module on `adata_sub`, then compare hub vs non-hub within each population.
   - Then summarize by coupling sign:
     - For positive-coupling lineages: do hubs show **simultaneous elevation** of multiple module scores?
     - For negative-coupling lineages: is there a **dominant module (e.g., contractile/VSMC)** with relatively weaker elevation of others?

3. **Use unsupervised structure within hub-upregulated genes separately for positive vs negative lineages.**
   - Take the union of hub-up genes in positive-coupling lineages, perform:
     - Hierarchical clustering or a PCA/tSNE/UMAP on hub cells only, restricted to these genes, to see whether hubs occupy a more **dispersed, multi-axis** space.
   - Repeat for negative-coupling lineages with their hub-up genes.
   - Compare:
     - Positive hubs might show **multiple correlated axes** (fibroblast–endothelial–developmental mixture).
     - Negative hubs might organize around one or two axes (e.g., VSMC contractile vs EPDC signaling), or vice versa. This will clarify whether “interface hub” in positive-coupling lineages truly reflects greater transcriptional complexity in a biological sense.

4. **Refine the functional annotation while staying distinct from the paper.**
   - Without external databases, you can still extend the module classification:
     - Utilize the **existing panel**: cluster genes by coexpression across all hubs, then label clusters as contractile-like, ECM-like, endothelial-like, developmental-like based on known cardiac markers already in the panel.
     - Then, for each lineage and coupling sign, calculate enrichment of cluster-specific scores in hubs vs non-hubs.
   - This could reveal that, for instance, negative-coupling EPDC/VSMC hubs are particularly enriched for one “VSMC-contractile” cluster plus a “vascular signaling” cluster, whereas positive-coupling hubs spread their weight across several more diverse clusters.

5. **Population-level structure of hub programs.**
   - Using your `sig` table, build a gene × population matrix of hub-upregulated (e.g., indicator or mean logFC).
   - Run hierarchical clustering on populations and genes:
     - Do positive-coupling lineages (vFibro/aFibro/adFibro/BEC/VEC/Pericyte/aEndocardial/vEndocardial) cluster together by hub program?
     - Do EPDC and VSMC hub programs cluster apart, and are they defined by the negative-only genes like `ARHGAP18`, `ALDH1A2`, `ADM`, `CASQ2`, `CKMT2`, `RGS5`?
   - This addresses the “systematically differ between lineages” part in a more global way than individual gene counts.

6. **Check whether hub-upregulation correlates with Complexity at single-cell level.**
   - For each population, correlate:
     - Complexity residual (after Purity/UMI) with expression of key hub markers (e.g., `CXCL12`, `FN1`, `POSTN`, `NR2F2` for positive; `ALDH1A2`, `ARHGAP18/29`, `CASQ2`, `CKMT2`, `RGS5` for negative).
   - In **positive-coupling** lineages, hub markers should correlate positively with Complexity residual; in **negative-coupling** lineages, the direction may be reversed for their distinctive markers.
   - This directly links the DE signature to the earlier coupling regression rather than relying solely on the hub label.

7. **Spatial context of hub programs, stratified by coupling sign.**
   - While your current step is non-spatial, the hypothesis is about interface hubs; it would be informative (in a separate step) to:
     - Map the expression of positive-only vs negative-only hub-up genes on the spatial coordinates within their respective lineages.
   - This can test whether positive-coupling hubs marked by genes like `CXCL12`, `FN1`, `POSTN`, `NPR3` sit at **multicellular interfaces** (fibroblast–endothelial–endocardial boundaries) versus EPDC/VSMC hubs expressing `ALDH1A2/ARHGAP18/29/CASQ2/CKMT2` being **perivascular or outer-layer–biased**.

Caveats and refinements
-----------------------

- The panel is limited and biased toward cardiac and vascular markers; both positive and negative lineages will inevitably hit all four coarse modules once you union enough hub-up genes. So the **binary presence/absence** module summary is too blunt to support or refute “broader, multi-module activation”.
- Your module definition is hard-coded and fixed; a more data-driven clustering of genes within this panel will allow a subtler distinction of hub programs.
- The current step is cross-population aggregation of hub-up genes; when you want to argue about “broad vs specialized,” it’s important to **normalize by panel size and per-gene detectability** and to compare distributions of module scores, not just membership.

Summary
-------

- The analysis validates that hubs in both positive- and negative-coupling lineages are transcriptionally distinct from non-hubs and that their programs **systematically differ in content**, with many negative-only vs positive-only hub genes.
- It does **not yet cleanly demonstrate** that positive-coupling hubs have broader multi-module activation; current module summaries show broad coverage for both.
- Next steps should focus on **quantitative module weighting, data-driven gene-module definitions, and single-cell-level module scores and clustering** split by coupling sign, to more rigorously test the “multi-module vs specialized” component of the hypothesis and to relate these hub signatures back to Complexity residuals and spatial interfaces.